# Lab Day 2 — Bài làm hoàn chỉnh trên Google Colab
**Hoàng Văn Nam — Mã học viên: 2A202602853**

Chọn **Runtime → Change runtime type → GPU**, chạy từ trên xuống hoặc Run all.
Notebook tự chứa toàn bộ code; không cần chạy local, clone repo hoặc upload module Python.
Đủ EDA, 5 backbone, ablation A–G khi phù hợp, nhiều phương pháp suy luận/latency, 3 seed final và baseline, Excel/báo cáo/ZIP.
Lưu checkpoint/kết quả trên Drive, tiếp tục khi Colab mất phiên. Chọn mọi thứ bằng val; khóa lựa chọn trước test.
Chưa điền trước số liệu huấn luyện; kết quả được sinh từ lần chạy của bạn.


## 0. Cài đặt môi trường

In [ ]:
import sys, os, json, platform, subprocess, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Chọn GPU trong Runtime → Change runtime type rồi chạy lại."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.30",
 "fvcore==0.1.5.post20221221", "openpyxl>=3.1,<4", "scipy>=1.11,<2",
 "scikit-learn>=1.4,<2", "pandas>=2,<3", "matplotlib>=3.8,<4", "Pillow>=10,<13"], check=True)
import numpy as np, pandas as pd, torchvision, timm
from IPython.display import display, Markdown, Image
STUDENT_ID, STUDENT_NAME = "2A202602853", "Hoàng Văn Nam"
USE_DRIVE = True
EPOCHS, BATCH_SIZE, NUM_WORKERS = 12, 32, 2
# Batch 32 giảm bộ nhớ so với 64 gợi ý; giữ cùng batch/epoch cho cả chiến dịch.
if USE_DRIVE:
 from google.colab import drive
 drive.mount("/content/drive")
 OUTPUT_DIR = Path("/content/drive/MyDrive/DeepWeeds_Lab/2A202602853_Hoang_Van_Nam")
else:
 OUTPUT_DIR = Path("/content/submissions/2A202602853_Hoang_Van_Nam")
WORK_DIR=Path("/content/deepweeds_lab")
(WORK_DIR/"code").mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
os.chdir(WORK_DIR)
sys.path.insert(0,str(WORK_DIR)); sys.path.insert(0,str(WORK_DIR/"code"))
print(STUDENT_ID, STUDENT_NAME, "GPU:", torch.cuda.get_device_name(0))
print("python",platform.python_version(),"torch",torch.__version__,"torchvision",torchvision.__version__,"timm",timm.__version__)
print("Output:",OUTPUT_DIR,"— giữ nguyên cấu hình và đường dẫn khi resume.")


### Mã nguồn tự chứa — chỉ tạo trong runtime Colab
Chạy các ô này trước EDA. Không cần upload file Python riêng. Có thể thu gọn sau khi đọc.


#### `dataset.py`

In [ ]:
%%writefile code/dataset.py
"""DeepWeeds validation and shared preprocessing."""
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

NUM_CLASSES = 9
CLASS_NAMES = ['Chinee apple', 'Lantana', 'Parkinsonia', 'Parthenium', 'Prickly acacia', 'Rubber vine', 'Siam weed', 'Snake weed', 'Negative']
IMAGENET_MEAN = (.485, .456, .406)
IMAGENET_STD = (.229, .224, .225)


def load_split(labels_dir, fold=0):
    if fold not in range(5):
        raise ValueError('fold must be 0..4')
    frames = []
    for split in ('train', 'val', 'test'):
        frame = pd.read_csv(Path(labels_dir) / f'{split}_subset{fold}.csv')
        if not {'Filename', 'Label'}.issubset(frame):
            raise ValueError(f'{split}: missing Filename/Label')
        frames.append(frame)
    return tuple(frames)


def check_split(train_df, val_df, test_df, images_dir):
    frames = dict(zip(('train', 'val', 'test'), (train_df, val_df, test_df)))
    names = {}
    for split, frame in frames.items():
        if frame.empty or frame[['Filename', 'Label']].isna().any().any():
            raise ValueError(f'{split}: empty split or missing values')
        if frame.Filename.duplicated().any():
            raise ValueError(f'{split}: duplicate filenames')
        if not frame.Label.isin(range(9)).all() or set(frame.Label) != set(range(9)):
            raise ValueError(f'{split}: invalid labels or missing classes')
        if any(Path(n).name != n or '\\' in n for n in frame.Filename):
            raise ValueError(f'{split}: Filename must be a basename')
        names[split] = set(frame.Filename)
    overlap = {f'{a}_{b}': len(names[a] & names[b]) for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test'))}
    if any(overlap.values()):
        raise ValueError(f'Split overlap: {overlap}')
    union = set.union(*names.values())
    if len(union) != 17509:
        raise ValueError(f'Expected 17509 distinct files, got {len(union)}')
    missing = sorted(n for n in union if not (Path(images_dir) / n).is_file())
    if missing:
        raise ValueError(f'Missing {len(missing)} images: {missing[:5]}')
    for split, expected in (('train', .6), ('val', .2), ('test', .2)):
        if abs(len(frames[split]) / len(union) - expected) > .01:
            raise ValueError(f'{split}: ratio differs by more than 1 percentage point')
    return {'n': {s: len(f) for s, f in frames.items()},
            'per_class': {s: f.Label.value_counts().reindex(range(9), fill_value=0).sort_index().to_dict() for s, f in frames.items()},
            'overlap': overlap, 'union': len(union), 'missing': 0}


def build_transforms(train, img_size=224, aug='basic', mean=IMAGENET_MEAN, std=IMAGENET_STD):
    if train:
        ops = [T.RandomResizedCrop(img_size), T.RandomHorizontalFlip()]
        extras = {'basic': [], 'color': [T.ColorJitter(.2, .2, .2, .05)], 'trivial': [T.TrivialAugmentWide()], 'randaug': [T.RandAugment()]}
        if aug not in extras:
            raise ValueError(f'Unknown augmentation: {aug}')
        ops += extras[aug]
    else:
        ops = [T.Resize(round(img_size * 256 / 224)), T.CenterCrop(img_size)]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])


def check_manifest(splits, labels_dir):
    """Audit original labels; preserve the author's fixed split (README S1).

    Upstream fold 0 assigns this one image to class 0, while labels.csv assigns
    class 1. Log this known discrepancy; reject any additional disagreement.
    """
    manifest = pd.read_csv(Path(labels_dir) / 'labels.csv')
    combined = pd.concat(splits, ignore_index=True)
    if manifest.Filename.duplicated().any() or len(manifest) != 17509:
        raise ValueError('Invalid labels.csv manifest')
    reference = manifest.set_index('Filename').Label
    if set(reference.index) != set(combined.Filename):
        raise ValueError('Split filenames differ from labels.csv')
    labels = reference.loc[combined.Filename].to_numpy()
    mismatches = combined.loc[labels != combined.Label.to_numpy(), ['Filename', 'Label']].copy()
    mismatches['manifest_label'] = labels[labels != combined.Label.to_numpy()]
    records = mismatches.to_dict('records')
    known = {'Filename': '20170714-110407-3.jpg', 'Label': 0, 'manifest_label': 1}
    if any(record != known for record in records):
        raise ValueError(f'Unexpected split label disagreement: {records[:5]}')
    return records


class DeepWeedsDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_dir, self.transform = Path(images_dir), transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(self.images_dir / row.Filename) as image:
            image = image.convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row.Label), row.Filename


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)


def make_loader(df, images_dir, transform, batch_size, train, sampler=None, num_workers=2):
    if sampler not in (None, 'balanced') or (sampler and not train):
        raise ValueError('Sampler must be None or balanced (train only)')
    generator = torch.Generator().manual_seed(torch.initial_seed())
    sampling = None
    if sampler == 'balanced':
        weights = 1 / df.Label.map(df.Label.value_counts()).to_numpy()
        sampling = WeightedRandomSampler(weights, len(df), replacement=True, generator=generator)
    return DataLoader(DeepWeedsDataset(df, images_dir, transform), batch_size=batch_size,
                      shuffle=train and sampling is None, sampler=sampling,
                      drop_last=train and len(df) >= batch_size, pin_memory=torch.cuda.is_available(),
                      num_workers=num_workers, worker_init_fn=seed_worker, generator=generator)


#### `model.py`

In [ ]:
%%writefile code/model.py
"""timm backbones and native AdamW groups."""
import torch
import timm

SUGGESTED_BACKBONES = {'resnet50': 'resnet50', 'resnext50': 'resnext50_32x4d', 'convnext_tiny': 'convnext_tiny', 'deit_small': 'deit_small_patch16_224', 'swin_tiny': 'swin_tiny_patch4_window7_224', 'efficientnet_b0': 'efficientnet_b0', 'mobilenetv3': 'mobilenetv3_large_100'}


def build_model(name, pretrained=True, num_classes=9, drop_rate=0., init='finetune'):
    if init not in ('scratch', 'frozen', 'finetune'):
        raise ValueError(f'Unknown init: {init}')
    model = timm.create_model(SUGGESTED_BACKBONES.get(name, name), pretrained=pretrained and init != 'scratch', num_classes=num_classes, drop_rate=drop_rate)
    if init == 'frozen':
        freeze_backbone(model)
    return model


def freeze_backbone(model):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    for parameter in model.get_classifier().parameters():
        parameter.requires_grad_(True)


def param_groups(model, lr_backbone, lr_head, weight_decay):
    head = {id(p) for p in model.get_classifier().parameters()}
    no_decay = model.no_weight_decay() if hasattr(model, 'no_weight_decay') else set()
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lr = lr_head if id(p) in head else lr_backbone
        decay = 0. if p.ndim <= 1 or name.endswith('.bias') or name in no_decay else weight_decay
        groups.setdefault((lr, decay), []).append(p)
    return [{'params': params, 'lr': lr, 'weight_decay': wd} for (lr, wd), params in groups.items()]


def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def count_gmacs(model, img_size=224):
    """Optional fvcore estimate; never silently report unsupported operations as zero."""
    from fvcore.nn import FlopCountAnalysis
    modes = {m: m.training for m in model.modules()}
    try:
        model.eval()
        counter = FlopCountAnalysis(model, torch.zeros(1, 3, img_size, img_size, device=next(model.parameters()).device))
        result = counter.total() / 1e9
        if counter.unsupported_ops():
            raise ValueError(f'Incomplete GMAC estimate: {dict(counter.unsupported_ops())}')
        return result
    finally:
        for module, training in modes.items():
            module.training = training


#### `losses.py`

In [ ]:
%%writefile code/losses.py
"""Loss and batch-mixing implementations."""
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


class LabelSmoothingCE(nn.CrossEntropyLoss):
    def __init__(self, smoothing=.1):
        super().__init__(label_smoothing=smoothing)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2., alpha=None):
        super().__init__()
        if gamma < 0:
            raise ValueError('gamma must be nonnegative')
        self.gamma = gamma
        self.register_buffer('alpha', None if alpha is None else torch.as_tensor(alpha, dtype=torch.float32))

    def forward(self, logits, target):
        log_pt = F.log_softmax(logits, dim=1).gather(1, target[:, None]).squeeze(1)
        loss = -(1 - log_pt.exp()).pow(self.gamma) * log_pt
        if self.alpha is not None:
            loss = loss * self.alpha[target]
        return loss.mean()


def build_criterion(kind='ce', **kw):
    if kind == 'ce':
        return nn.CrossEntropyLoss()
    if kind == 'ls':
        return LabelSmoothingCE(kw.get('smoothing', .1))
    if kind == 'focal':
        return FocalLoss(kw.get('gamma', 2.), kw.get('alpha'))
    if kind == 'ce_weighted':
        if kw.get('weight') is None:
            raise ValueError('ce_weighted requires train-only class weights')
        return nn.CrossEntropyLoss(weight=kw['weight'])
    raise ValueError(f'Unknown loss: {kind}')


def class_weights(counts, beta=0.):
    counts = torch.as_tensor(counts, dtype=torch.float64)
    if (counts <= 0).any() or not 0 <= beta < 1:
        raise ValueError('Positive counts and 0 <= beta < 1 required')
    weights = 1 / counts if beta == 0 else (1 - beta) / (-torch.expm1(counts * np.log(beta)))
    return (weights / weights.mean()).float()


def mix_batch(x, y, alpha=1., mode='cutmix'):
    if alpha <= 0 or mode not in ('mixup', 'cutmix'):
        raise ValueError('Positive alpha and mode mixup/cutmix required')
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(len(x), device=x.device)
    if mode == 'mixup':
        mixed = lam * x + (1 - lam) * x[perm]
    else:
        h, w = x.shape[-2:]
        cut_w, cut_h = int(w * np.sqrt(1 - lam)), int(h * np.sqrt(1 - lam))
        cx, cy = np.random.randint(w), np.random.randint(h)
        x1, x2 = max(cx - cut_w // 2, 0), min(cx + (cut_w + 1) // 2, w)
        y1, y2 = max(cy - cut_h // 2, 0), min(cy + (cut_h + 1) // 2, h)
        mixed = x.clone()
        mixed[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1 - (x2 - x1) * (y2 - y1) / (w * h)
    return mixed, (y, y[perm], lam)


def mixed_loss(criterion, logits, targets):
    a, b, lam = targets
    return lam * criterion(logits, a) + (1 - lam) * criterion(logits, b)


#### `train.py`

In [ ]:
%%writefile code/train.py
"""One configurable train loop. Test inference stays off until the final stage."""
from __future__ import annotations
import argparse
import hashlib
import copy
import json
import math
import random
import sys
import time
import platform
from dataclasses import asdict, dataclass, fields
from pathlib import Path
from typing import get_type_hints, get_args
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from eval import compute_metrics, save_predictions
import dataset
import model as models
import losses


@dataclass
class Config:
    exp_id: str = 'T00'
    seed: int = 0
    fold: int = 0
    backbone: str = 'resnet50'
    init: str = 'finetune'
    drop_rate: float = 0.
    img_size: int = 224
    aug: str = 'basic'
    sampler: str | None = None
    mix: str | None = None
    mix_alpha: float = 1.
    loss: str = 'ce'
    label_smoothing: float = 0.
    focal_gamma: float = 2.
    class_weight_beta: float | None = None
    epochs: int = 12
    batch_size: int = 64
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = .05
    warmup_epochs: float = 1.
    ema_decay: float | None = None
    amp: bool = True
    num_workers: int = 2
    images_dir: str = 'data/images'
    labels_dir: str = 'data/labels'
    out_dir: str = 'runs'
    pred_dir: str = 'predictions'
    save_test_predictions: bool = False
    resume: bool = False
    curves_dir: str = 'curves'


def run_dir(cfg):
    return Path(cfg.out_dir) / cfg.exp_id / f'seed{cfg.seed}'


def pred_path(cfg, split):
    return Path(cfg.pred_dir) / f'{cfg.exp_id}_seed{cfg.seed}_{split}.csv'


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def validate_resume_config(cfg, saved):
    current = asdict(cfg)
    ignored = {'resume'}
    if any(current.get(k) != v for k, v in saved.items() if k not in ignored):
        raise ValueError('Resume configuration differs from saved experiment')


def save_checkpoint(state, path):
    temporary = Path(path).with_suffix('.tmp')
    torch.save(state, temporary)
    temporary.replace(path)


def build_optimizer(model, cfg):
    return torch.optim.AdamW(models.param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay))


def build_scheduler(optimizer, cfg, steps_per_epoch):
    if cfg.epochs <= 0 or steps_per_epoch <= 0 or not 0 <= cfg.warmup_epochs < cfg.epochs:
        raise ValueError('Positive epochs/steps and 0 <= warmup_epochs < epochs required')
    total = cfg.epochs * steps_per_epoch
    warmup = round(cfg.warmup_epochs * steps_per_epoch)
    def factor(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        progress = min(1., (step - warmup) / max(1, total - warmup))
        return .5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


class EMA:
    def __init__(self, model, decay):
        if not 0 <= decay < 1:
            raise ValueError('EMA decay must be in [0,1)')
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        self.model.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        source = model.state_dict()
        for name, target in self.model.state_dict().items():
            if target.is_floating_point():
                target.lerp_(source[name], 1 - self.decay)
            else:
                target.copy_(source[name])


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg, device, ema=None):
    model.train()
    if cfg.init == 'frozen':
        model.eval()  # includes BN/dropout in the frozen backbone
        model.get_classifier().train()
    total, n = 0., 0
    device = torch.device(device)
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        target = y
        if cfg.mix:
            x, target = losses.mix_batch(x, y, cfg.mix_alpha, cfg.mix)
        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == 'cuda'):
            logits = model(x)
            loss = losses.mixed_loss(criterion, logits, target) if cfg.mix else criterion(logits, target)
        if not torch.isfinite(loss):
            raise ValueError('Non-finite training loss')
        scaler.scale(loss).backward()
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= scale_before:  # no scheduler/EMA update after an AMP-skipped step
            scheduler.step()
            if ema is not None:
                ema.update(model)
        total += float(loss.detach()) * len(y)
        n += len(y)
    if not n:
        raise ValueError('Empty training loader')
    return {'train_loss': total / n, 'lr': optimizer.param_groups[0]['lr']}


@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    names, truth, outputs, total, n = [], [], [], 0., 0
    for x, y, filenames in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        if not torch.isfinite(loss):
            raise ValueError('Non-finite evaluation loss')
        total += float(loss) * len(y)
        n += len(y)
        names.extend(filenames)
        truth.append(y.cpu().numpy())
        outputs.append(logits.float().cpu().numpy())
    if not n:
        raise ValueError('Empty evaluation loader')
    return names, np.concatenate(truth), np.concatenate(outputs), total / n


def plot_curves(history, path, title):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(frame.epoch, frame.train_loss, label='train')
    axes[0].plot(frame.epoch, frame.val_loss, label='val')
    axes[0].set(xlabel='Epoch', ylabel='Loss')
    axes[0].legend()
    axes[1].plot(frame.epoch, frame.macro_f1)
    axes[1].set(xlabel='Epoch', ylabel='Val macro-F1')
    fig.suptitle(title)
    fig.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


def run(cfg):
    if Path(cfg.exp_id).name != cfg.exp_id or not cfg.exp_id:
        raise ValueError('exp_id must be a non-empty basename')
    if cfg.batch_size < 2 or cfg.epochs < 1:
        raise ValueError('batch_size >= 2 and epochs >= 1 required')
    folder = run_dir(cfg)
    if cfg.resume and folder.exists():
        saved = json.loads((folder / 'config.json').read_text(encoding='utf-8'))['config']
        validate_resume_config(cfg, saved)
        if (folder / 'summary.json').exists():
            return json.loads((folder / 'summary.json').read_text(encoding='utf-8'))
        # Before the first committed epoch, restart that epoch from the same seed.
    elif folder.exists() or pred_path(cfg, 'val').exists() or (cfg.save_test_predictions and pred_path(cfg, 'test').exists()):
        raise FileExistsError('Run/predictions already exist; use a new exp_id, do not overwrite final test evidence')
    set_seed(cfg.seed)
    splits = dataset.load_split(cfg.labels_dir, cfg.fold)
    check = dataset.check_split(*splits, cfg.images_dir)
    check['manifest_label_mismatches'] = dataset.check_manifest(splits, cfg.labels_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = models.build_model(cfg.backbone, init=cfg.init, drop_rate=cfg.drop_rate).to(device)
    from timm.data import resolve_model_data_config
    data_config = resolve_model_data_config(model)
    mean, std = data_config['mean'], data_config['std']
    train_loader = dataset.make_loader(splits[0], cfg.images_dir, dataset.build_transforms(True, cfg.img_size, cfg.aug, mean, std), cfg.batch_size, True, cfg.sampler, cfg.num_workers)
    val_loader = dataset.make_loader(splits[1], cfg.images_dir, dataset.build_transforms(False, cfg.img_size, mean=mean, std=std), cfg.batch_size, False, num_workers=cfg.num_workers)
    counts = splits[0].Label.value_counts().reindex(range(9)).to_numpy()
    weight = losses.class_weights(counts, cfg.class_weight_beta or 0.) if cfg.loss == 'ce_weighted' else None
    criterion = losses.build_criterion(cfg.loss, smoothing=cfg.label_smoothing, gamma=cfg.focal_gamma, weight=weight).to(device)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp and device.type == 'cuda')
    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay is not None else None
    folder.mkdir(parents=True, exist_ok=cfg.resume)
    metadata = {'config': asdict(cfg), 'pretrained_cfg': model.pretrained_cfg,
                'source_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob('*.py')},
                'label_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(cfg.labels_dir).glob('*.csv')},
                'preprocessing': {'mean': mean, 'std': std, 'eval_resize': round(cfg.img_size * 256 / 224), 'crop': cfg.img_size},
                'device': str(device), 'amp_active': scaler.is_enabled(), 'split_check': check,
                'torch': torch.__version__, 'python': platform.python_version(),
                'numpy': np.__version__, 'pandas': pd.__version__}
    import timm
    import torchvision
    metadata.update({'timm': timm.__version__, 'torchvision': torchvision.__version__,
                     'gpu': torch.cuda.get_device_name() if device.type == 'cuda' else None,
                     'scheduler': 'per optimizer step, linear warmup then cosine'})
    (folder / 'config.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    history, best, best_epoch, start_epoch = [], -1., 0, 1
    if cfg.resume and (folder / 'last.pt').exists():
        # Only load trusted checkpoints created locally by this training code.
        state = torch.load(folder / 'last.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        if ema:
            ema.model.load_state_dict(state['ema'])
        history, best, best_epoch = state['history'], state['best'], state['best_epoch']
        start_epoch = state['epoch'] + 1
        random.setstate(state['random'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        train_loader.generator.set_state(state['loader_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
    for epoch in range(start_epoch, cfg.epochs + 1):
        started = time.perf_counter()
        stats = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, cfg, device, ema)
        candidate = ema.model if ema else model
        _, y, logits, val_loss = evaluate(candidate, val_loader, criterion, device)
        probs = torch.from_numpy(logits).softmax(1).numpy()
        metrics = compute_metrics(y, probs.argmax(1), probs)
        history.append({'epoch': epoch, **stats, 'val_loss': val_loss, 'macro_f1': metrics['macro_f1'], 'top1': metrics['top1'], 'seconds': time.perf_counter() - started})
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        if metrics['macro_f1'] > best:
            best, best_epoch = metrics['macro_f1'], epoch
            save_checkpoint({'model': candidate.state_dict(), 'epoch': epoch, 'macro_f1': best}, folder / 'best.pt')
        save_checkpoint({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                         'ema': ema.model.state_dict() if ema else None, 'epoch': epoch,
                         'history': history, 'best': best, 'best_epoch': best_epoch,
                         'random': random.getstate(), 'numpy_rng': np.random.get_state(),
                         'torch_rng': torch.get_rng_state(), 'loader_rng': train_loader.generator.get_state(),
                         'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, folder / 'last.pt')
        print(json.dumps(history[-1]), flush=True)
    model.load_state_dict(torch.load(folder / 'best.pt', map_location=device, weights_only=True)['model'])
    for split, frame in [('val', splits[1])] + ([('test', splits[2])] if cfg.save_test_predictions else []):
        loader = val_loader if split == 'val' else dataset.make_loader(frame, cfg.images_dir, dataset.build_transforms(False, cfg.img_size, mean=mean, std=std), cfg.batch_size, False, num_workers=cfg.num_workers)
        names, y, logits, _ = evaluate(model, loader, criterion, device)
        np.savez_compressed(folder / f'{split}_logits.npz', filenames=names, y_true=y, logits=logits)
        save_predictions(pred_path(cfg, split), names, y, torch.from_numpy(logits).softmax(1).numpy())
    plot_curves(history, Path(cfg.curves_dir) / f'{cfg.exp_id}_seed{cfg.seed}.png', f'{cfg.exp_id} {cfg.backbone} seed {cfg.seed}')
    summary = {'best_epoch': best_epoch, 'macro_f1_val': best, 'params_m': models.count_params(model), 'gmacs': None,
               'top1_val': next(h['top1'] for h in history if h['epoch'] == best_epoch),
               'gmacs_note': 'Not measured; optional fvcore count_gmacs', 'mean_epoch_seconds': np.mean([h['seconds'] for h in history])}
    try:
        summary['gmacs'] = models.count_gmacs(model, cfg.img_size)
        summary['gmacs_note'] = 'fvcore estimate; one multiply-add counted as one operation'
    except (ImportError, ValueError) as error:
        summary['gmacs_note'] = str(error)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary


def parse_overrides(pairs):
    hints = get_type_hints(Config)
    result = {}
    for pair in pairs:
        if '=' not in pair:
            raise ValueError(f'Expected KEY=VALUE: {pair}')
        key, value = pair.split('=', 1)
        if key not in hints:
            raise ValueError(f'Unknown Config field: {key}')
        args = get_args(hints[key])
        if value.lower() == 'none' and type(None) in args:
            result[key] = None
            continue
        kind = next((t for t in args if t is not type(None)), hints[key])
        if kind is bool:
            if value.lower() not in ('true', 'false'):
                raise ValueError(f'{key}: use true/false')
            result[key] = value.lower() == 'true'
        else:
            result[key] = kind(value)
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--set', nargs='*', default=[])
    args = parser.parse_args()
    print(json.dumps(run(Config(**parse_overrides(args.set))), indent=2))


if __name__ == '__main__':
    main()


#### `inference.py`

In [ ]:
%%writefile code/inference.py
"""inference.py - các phương pháp suy luận (Bước 3 của GUIDE.md).

Cài đặt hoàn chỉnh cho Colab.
Liên hệ slide Day 2: TTA (trang 62-66, 75), ensemble/EMA/soup (trang 67), độ phân giải kiểm tra
(trang 68), temperature scaling (trang 69), gộp BatchNorm (trang 71).

Mọi hàm phải chạy ở chế độ eval, không gradient. Chọn phương pháp CHỈ dựa trên val;
nhiệt độ T khớp trên VAL rồi áp dụng sang test (README.md, S2 và S4).

Giao diện bạn nên giữ:
    predict_logits(model, loader, device, view=None) -> (filenames, y_true, logits[N, 9])
    aggregate_views(list_of_logits, space)           -> probs[N, 9]
    fit_temperature(val_logits, val_labels)          -> float T
    apply_temperature(logits, T)                     -> probs
    ensemble_probs(list_of_probs)                    -> probs
    fuse_conv_bn(model)                              -> model (BN đã gộp vào conv)
"""
from __future__ import annotations
import copy
import numpy as np
import torch
import torch.nn.functional as F
from scipy.optimize import minimize_scalar


def predict_logits(model, loader, device, view=None):
    """Chạy model trên loader và gom logit theo đúng thứ tự file.

    `view` là hàm biến đổi batch ảnh trước khi đưa vào model (ví dụ lật ngang), hoặc None.
    Thực hiện: model.eval(), torch.inference_mode(), (tuỳ chọn) autocast. Trả về numpy.
    """
    model.eval()
    names, labels, outputs = [], [], []
    with torch.inference_mode():
        for x, y, filenames in loader:
            x = x.to(device)
            logits = model(view(x) if view else x)
            names.extend(filenames)
            labels.append(np.asarray(y))
            outputs.append(logits.float().cpu().numpy())
    if not outputs:
        raise ValueError('Empty loader')
    result = np.concatenate(outputs)
    if not np.isfinite(result).all():
        raise ValueError('Non-finite logits')
    return names, np.concatenate(labels), result


def view_identity(x):
    return x


def view_hflip(x):
    """Lật ngang batch (N, C, H, W). Thực hiện: dùng torch.flip trên chiều rộng (slide trang 75)."""
    return torch.flip(x, [-1])


def views_multicrop(x, crop: int):
    """5 crop (4 góc + giữa) kích thước `crop`, và tuỳ chọn thêm bản lật. Trả về list các batch. Đã cài đặt."""
    h, w = x.shape[-2:]
    if not 0 < crop <= min(h, w):
        raise ValueError('Crop must fit input')
    positions = [(0, 0), (0, w-crop), (h-crop, 0), (h-crop, w-crop), ((h-crop)//2, (w-crop)//2)]
    return [x[..., top:top+crop, left:left+crop] for top, left in positions]


def views_multiscale(x, sizes):
    """Resize batch về từng kích thước trong `sizes`, trả về list các batch. Đã cài đặt.

    Lưu ý: model phải chấp nhận ảnh khác kích thước lúc train (CNN có global pooling thì được;
    ViT/Swin cần xử lý riêng vị trí/cửa sổ). Ghi rõ giới hạn bạn gặp.
    """
    if not sizes or any(s <= 0 for s in sizes):
        raise ValueError('Positive sizes required')
    return [F.interpolate(x, size=(s, s), mode='bilinear', align_corners=False) for s in sizes]


def aggregate_views(logits_per_view, space: str = "prob"):
    """Gộp K lượt chạy của TTA thành một dự đoán (slide trang 62).

      - space="prob":  trung bình softmax của từng view
      - space="logit": trung bình logit rồi softmax
    Slide chưa kết luận cách nào luôn tốt hơn: chọn một và ghi rõ, hoặc so sánh cả hai (I03).
    Thực hiện: trả về xác suất (N, 9) đã chuẩn hoá.
    """
    if not logits_per_view or len({np.asarray(z).shape for z in logits_per_view}) != 1:
        raise ValueError('Non-empty, aligned views required')
    if space == 'prob':
        return np.mean([apply_temperature(z, 1.) for z in logits_per_view], axis=0)
    if space == 'logit':
        return apply_temperature(np.mean(logits_per_view, axis=0), 1.)
    raise ValueError('space must be prob or logit')


def ensemble_probs(list_of_probs):
    """Trung bình xác suất của nhiều mô hình (khác backbone hoặc khác seed). Đã cài đặt.

    Chi phí suy luận = số mô hình. Chỉ ghép các mô hình trên CÙNG tập ảnh và cùng thứ tự file.
    """
    if not list_of_probs or len({np.asarray(p).shape for p in list_of_probs}) != 1:
        raise ValueError('Non-empty, aligned predictions required')
    for p in list_of_probs:
        if not np.isfinite(p).all() or (np.asarray(p) < 0).any() or not np.allclose(np.sum(p, axis=1), 1):
            raise ValueError('Normalized probabilities required')
    return np.mean(list_of_probs, axis=0)


def fit_temperature(val_logits, val_labels) -> float:
    """Tìm nhiệt độ T > 0 cực tiểu NLL trên VAL: p = softmax(logit / T)  (slide trang 69).

    Thực hiện: tối ưu hoá một tham số (LBFGS trên log T, hoặc tìm lưới thô rồi tinh).
    Accuracy không đổi vì thứ tự lớp không đổi. KHÔNG khớp T trên test.
    """
    z, y = np.asarray(val_logits, dtype=np.float64), np.asarray(val_labels)
    if z.ndim != 2 or len(z) == 0 or y.shape != (len(z),) or not np.isfinite(z).all():
        raise ValueError('Invalid validation logits/labels')
    if not np.issubdtype(y.dtype, np.integer) or (y < 0).any() or (y >= z.shape[1]).any():
        raise ValueError('Invalid labels')
    def nll(log_t):
        scaled = z / np.exp(log_t)
        scaled -= scaled.max(axis=1, keepdims=True)
        return np.mean(np.log(np.exp(scaled).sum(axis=1)) - scaled[np.arange(len(y)), y])
    result = minimize_scalar(nll, bounds=(-5., 5.), method='bounded')
    if not result.success:
        raise RuntimeError('Temperature fit failed')
    return float(np.exp(result.x)) if result.fun < nll(0.) else 1.


def apply_temperature(logits, T: float):
    """Trả về softmax(logits / T). Đã cài đặt."""
    z = np.asarray(logits, dtype=np.float64)
    if z.ndim != 2 or not np.isfinite(z).all() or not np.isfinite(T) or T <= 0:
        raise ValueError('Finite logits and positive temperature required')
    z = z / T
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)


def fuse_conv_bn(model):
    """Gộp BatchNorm vào tích chập liền trước, chính xác lúc suy luận (slide trang 71, 75):

        w' = gamma * w / sqrt(var + eps)        b' = beta + gamma * (b - mean) / sqrt(var + eps)

    Thực hiện:
      - model.eval() trước
      - với từng cặp (Conv2d, BatchNorm2d) liền kề: tạo conv mới (có bias) và thay BN bằng Identity
      - kiểm tra: đầu ra trước/sau gộp lệch nhau cỡ 1e-5 trở xuống (in ra sai số lớn nhất)
    Với kiến trúc không có BN (ViT, Swin, ConvNeXt dùng LayerNorm), mục này không áp dụng; ghi rõ.
    """
    fused = copy.deepcopy(model).eval()
    # Restrict fusion to known forward-adjacent patterns, never arbitrary siblings.
    from timm.models.resnet import BasicBlock, Bottleneck, ResNet
    from timm.models._efficientnet_blocks import ConvBnAct, DepthwiseSeparableConv, InvertedResidual
    patterns = {ResNet: [('conv1', 'bn1')], BasicBlock: [('conv1', 'bn1'), ('conv2', 'bn2')],
                Bottleneck: [('conv1', 'bn1'), ('conv2', 'bn2'), ('conv3', 'bn3')],
                ConvBnAct: [('conv', 'bn1')],
                DepthwiseSeparableConv: [('conv_dw', 'bn1'), ('conv_pw', 'bn2')],
                InvertedResidual: [('conv_pw', 'bn1'), ('conv_dw', 'bn2'), ('conv_pwl', 'bn3')]}
    for module in fused.modules():
        pairs = []
        if isinstance(module, torch.nn.Sequential):
            children = list(module._modules.items())
            pairs = [(a, b) for (a, _), (b, _) in zip(children, children[1:])]
        pairs += patterns.get(type(module), [])
        for a, b in pairs:
            conv, bn = getattr(module, a), getattr(module, b)
            # timm BatchNormAct2d contains an activation: preserve that activation.
            if isinstance(conv, torch.nn.Conv2d) and isinstance(bn, torch.nn.BatchNorm2d):
                setattr(module, a, torch.nn.utils.fuse_conv_bn_eval(conv, bn))
                activation = copy.deepcopy(getattr(bn, 'act', torch.nn.Identity()))
                drop = copy.deepcopy(getattr(bn, 'drop', torch.nn.Identity()))
                setattr(module, b, torch.nn.Sequential(drop, activation))
    return fused


#### `benchmark.py`

In [ ]:
%%writefile code/benchmark.py
"""benchmark.py - đo độ trễ suy luận đúng cách (slide Day 2, trang 73 và 75; GUIDE.md mục 4.1).

Cài đặt hoàn chỉnh cho Colab.

Quy tắc đo (vi phạm bị trừ điểm, RUBRIC mục 3):
  - warmup: bỏ >= 10 lần chạy đầu
  - đồng bộ GPU: torch.cuda.synchronize() (hoặc CUDA event) TRƯỚC và SAU đoạn cần đo
  - >= 50 lần đo, báo cáo p50, p95, p99 (không chỉ trung bình)
  - ghi rõ GPU, dtype (FP32/AMP/FP16), batch, độ phân giải, có/không gộp BN, phiên bản torch
  - chọn và ghi rõ có tính tiền xử lý hay không
"""
from __future__ import annotations
import copy
import platform
import time
import numpy as np
import torch


def bench(fn, warmup: int = 10, iters: int = 100, sync=None) -> dict:
    """Đo thời gian một hàm `fn()` (không tham số), trả về mili-giây.

    `sync` là hàm đồng bộ (ví dụ torch.cuda.synchronize) hoặc None trên CPU.

    Thực hiện:
      - chạy warmup lần đầu rồi bỏ
      - với mỗi lần đo: sync(); t0 = time.perf_counter(); fn(); sync(); lấy hiệu * 1000
      - trả về {"p50": ..., "p95": ..., "p99": ..., "mean": ..., "n": iters}
    Gợi ý: dùng numpy.percentile hoặc torch.quantile.
    """
    if warmup < 10 or iters < 50:
        raise ValueError('At least 10 warmup and 50 timed iterations required')
    for _ in range(warmup):
        fn()
    timings = []
    for _ in range(iters):
        if sync:
            sync()
        start = time.perf_counter()
        fn()
        if sync:
            sync()
        timings.append((time.perf_counter() - start) * 1000)
    return dict(zip(('p50', 'p95', 'p99'), map(float, np.percentile(timings, [50, 95, 99]))),
                mean=float(np.mean(timings)), n=iters)


def latency_report(model, batch_size: int, img_size: int, dtype: str = "fp32", device: str = "cuda",
                   warmup: int = 10, iters: int = 100) -> dict:
    """Đo độ trễ forward của `model` với đầu vào ngẫu nhiên (batch_size, 3, img_size, img_size).

    Trả về dict có thể ghi thẳng vào sheet `Latency` của results.xlsx:
        {"gpu": ..., "dtype": ..., "batch": ..., "img_size": ..., "p50": ..., "p95": ..., "p99": ...,
         "images_per_s": batch_size / (p50 / 1000), "torch": torch.__version__}

    Thực hiện:
      - model.eval(), torch.inference_mode()
      - dtype: "fp32" | "amp" (autocast) | "fp16" (model.half())
      - gọi bench(...) với sync phù hợp; lấy tên GPU bằng torch.cuda.get_device_name
      - Nhớ: ở batch 1, AMP có thể CHẬM hơn FP32 (slide trang 73): đo thật, đừng giả định
    """
    return _latency(model, batch_size, img_size, dtype, device, warmup, iters, 1)


def tta_latency(model, k_views: int, **kw) -> dict:
    """Độ trễ của TTA K view: xấp xỉ K lần một lượt chạy (slide trang 63). Thực hiện: đo thật, so với K * p50."""
    if k_views < 1:
        raise ValueError('Positive number of views required')
    return _latency(model, k_views=k_views, **kw)


def _latency(model, batch_size=1, img_size=224, dtype='fp32', device='cuda',
             warmup=10, iters=100, k_views=1):
    device = torch.device(device)
    if dtype not in ('fp32', 'amp', 'fp16') or batch_size < 1 or img_size < 1:
        raise ValueError('Invalid dtype/batch/image size')
    candidate = copy.deepcopy(model).to(device).eval()
    candidate = candidate.half() if dtype == 'fp16' else candidate.float()
    x = torch.randn(batch_size, 3, img_size, img_size, device=device,
                    dtype=torch.float16 if dtype == 'fp16' else torch.float32)
    def forward():
        with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=dtype == 'amp'):
            for _ in range(k_views):
                candidate(x)
    sync = (lambda: torch.cuda.synchronize(device)) if device.type == 'cuda' else None
    result = bench(forward, warmup, iters, sync)
    result.update(gpu=torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU ' + platform.processor(),
                  dtype=dtype, batch=batch_size, img_size=img_size, k_views=k_views,
                  images_per_s=batch_size / (result['p50'] / 1000), torch=str(torch.__version__),
                  preprocessing=False, bn_fused=False)
    return result


#### `eda.py`

In [ ]:
%%writefile code/eda.py
"""Inspect image metadata in its original mode, without fitting preprocessing."""
from collections import Counter
from pathlib import Path
from PIL import Image


def scan_images(filenames, images_dir):
    sizes, modes, channels = Counter(), Counter(), Counter()
    for filename in filenames:
        with Image.open(Path(images_dir) / filename) as image:
            sizes[f'{image.width}x{image.height}'] += 1
            modes[image.mode] += 1
            channels[str(len(image.getbands()))] += 1
            image.verify()
    return {'sizes': dict(sizes), 'modes': dict(modes), 'channels': dict(channels),
            'verified_images': sum(sizes.values())}


#### `preflight.py`

In [ ]:
%%writefile code/preflight.py
"""Overfit a fixed batch, checking both training and evaluation modes."""
import torch


def overfit_batch(model, x, y, max_steps=100, lr=.001):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0)
    history = []
    for step in range(1, max_steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(x)
            eval_loss = float(criterion(logits, y))
            accuracy = float((logits.argmax(1) == y).float().mean())
        history.append({'step': step, 'train_loss': float(loss.detach()), 'eval_loss': eval_loss})
        if float(loss.detach()) < .02 and eval_loss < .05 and accuracy == 1.:
            break
    return {'steps': step, 'train_loss': float(loss.detach()), 'eval_loss': eval_loss,
            'accuracy': accuracy, 'history': history}


#### `prepare.py`

In [ ]:
%%writefile code/prepare.py
"""GUIDE 1.2-1.3: real-data EDA and a CPU-friendly pipeline sanity check.

Run from the repository root: python code/prepare.py
This checks train images only for overfitting; it never evaluates test predictions.
"""
import argparse
from dataclasses import asdict
import json
import math
from pathlib import Path
import platform

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import timm

import dataset
import model as models
import train
from eda import scan_images
from preflight import overfit_batch

PAPER_COUNTS = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]


def gallery(frame, images_dir, path, transform=None, samples=3):
    fig, axes = plt.subplots(9, samples, figsize=(3 * samples, 24))
    for label in range(9):
        rows = frame[frame.Label == label].head(samples)
        for ax, (_, row) in zip(axes[label], rows.iterrows()):
            with Image.open(Path(images_dir) / row.Filename) as image:
                image = image.convert('RGB')
            if transform is not None:
                image = transform(image)
                image = (image * torch.tensor(dataset.IMAGENET_STD)[:, None, None]
                         + torch.tensor(dataset.IMAGENET_MEAN)[:, None, None]).clamp(0, 1)
                image = image.permute(1, 2, 0).numpy()
            ax.imshow(image)
            ax.set_title(f'{dataset.CLASS_NAMES[label]}\n{row.Filename}', fontsize=8)
            ax.axis('off')
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def eda(images_dir, labels_dir, out_dir):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    splits = dataset.load_split(labels_dir)
    checks = dataset.check_split(*splits, images_dir)
    combined = pd.concat(splits, ignore_index=True)
    checks['manifest_label_mismatches'] = dataset.check_manifest(splits, labels_dir)
    counts = pd.DataFrame({name: frame.Label.value_counts().reindex(range(9), fill_value=0)
                           for name, frame in zip(('train', 'val', 'test'), splits)})
    counts['total'] = counts.sum(axis=1)
    counts['paper'] = PAPER_COUNTS
    counts['difference'] = counts.total - counts.paper
    counts.insert(0, 'class', dataset.CLASS_NAMES)
    counts.to_csv(out / 'class_counts.csv', index_label='label')
    image_stats = scan_images(combined.Filename, images_dir)
    checks.update({**image_stats,
                   'imbalance_ratio': float(counts.total.max() / counts.total.min()),
                   'train_imbalance_ratio': float(counts.train.max() / counts.train.min()),
                   'paper_counts_match': bool((counts.difference == 0).all())})
    (out / 'split_checks.json').write_text(json.dumps(checks, indent=2), encoding='utf-8')
    ax = counts.set_index('class')[['train', 'val', 'test']].plot.bar(figsize=(12, 5))
    ax.set(xlabel='Class', ylabel='Images', title='DeepWeeds fold 0: fixed splits')
    ax.figure.tight_layout()
    ax.figure.savefig(out / 'class_distribution.png', dpi=150)
    plt.close(ax.figure)
    gallery(splits[0], images_dir, out / 'class_samples.png')
    # No normalization statistics are fitted on val/test: use fixed ImageNet constants.
    lines = ['# EDA — DeepWeeds fold 0', '',
             '| Class | Train | Val | Test | Total | Paper | Difference |',
             '|---|---:|---:|---:|---:|---:|---:|']
    for _, row in counts.iterrows():
        lines.append('| ' + ' | '.join(str(row[c]) for c in counts.columns) + ' |')
    lines += ['', f'Split sizes: {checks["n"]}. Union: {checks["union"]}; overlaps: {checks["overlap"]}; missing: 0.',
              f'All {len(combined)} images passed PIL verification. Sizes: {checks["sizes"]}; modes: {checks["modes"]}; channels: {checks["channels"]}.',
              f'Largest/smallest class: {checks["imbalance_ratio"]:.4f} overall; {checks["train_imbalance_ratio"]:.4f} on train.',
              f'Counts match the supplied Table 1 reference: {checks["paper_counts_match"]}.', '',
              f'Original split vs labels.csv label differences: {checks["manifest_label_mismatches"]}. Keep the original split labels unchanged (S1).', '',
              '![Distribution](class_distribution.png)', '',
              'Three original TRAIN images per class; filenames and class labels are shown:', '',
              '![Train samples](class_samples.png)', '',
              'Input recipe: RGB, train RandomResizedCrop(224) + horizontal flip; val/test Resize(256) + CenterCrop(224).',
              'Use the pretrained weight mean/std (ImageNet defaults for the baseline). Never estimate normalization from test.']
    (out / 'eda.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return checks


def pipeline_check(images_dir, labels_dir, out_dir, backbone='resnet18', img_size=64, max_steps=100):
    """Overfit 9 real train images with a fresh scratch model; not a baseline experiment."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    train.set_seed(0)
    train_frame, _, _ = dataset.load_split(labels_dir)
    frame = train_frame.groupby('Label', sort=True).head(1).sort_values('Label')
    fixed = dataset.DeepWeedsDataset(frame, images_dir, dataset.build_transforms(False, img_size))
    x = torch.stack([fixed[i][0] for i in range(len(fixed))])
    y = torch.tensor(frame.Label.to_numpy(), dtype=torch.long)
    network = models.build_model(backbone, pretrained=False, init='scratch')
    criterion = torch.nn.CrossEntropyLoss()
    network.eval()
    with torch.no_grad():
        initial_loss = float(criterion(network(x), y))
    if abs(initial_loss - math.log(9)) > .5:
        raise AssertionError(f'Initial CE {initial_loss} far from ln(9)')
    overfit = overfit_batch(network, x, y, max_steps=max_steps)
    history = overfit['history']
    final_loss, accuracy = overfit['eval_loss'], overfit['accuracy']
    result = {'seed': 0, 'backbone': backbone, 'init': 'scratch', 'img_size': img_size,
              'device': 'cpu', 'samples': frame.Filename.tolist(), 'labels': frame.Label.tolist(),
              'initial_loss': initial_loss, 'expected_initial_loss': math.log(9),
              'steps': overfit['steps'], 'final_loss': final_loss, 'accuracy': accuracy,
              'passed': final_loss < .05 and accuracy == 1.,
              'note': 'Sanity check only; no pretrained baseline or validation/test experiment.'}
    pd.DataFrame(history).to_csv(out / 'overfit_history.csv', index=False)
    (out / 'pipeline_check.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    gallery(train_frame, images_dir, out / 'augmented_samples.png', dataset.build_transforms(True))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([h['step'] for h in history], [h['train_loss'] for h in history], label='train mode')
    ax.plot([h['step'] for h in history], [h['eval_loss'] for h in history], label='eval mode')
    ax.set(xlabel='Optimization step', ylabel='CE loss', title='Overfit 9 real TRAIN images')
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / 'overfit_curve.png', dpi=150)
    plt.close(fig)
    if not result['passed']:
        raise AssertionError(f'Batch did not overfit: {result}')
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--images-dir', default='data/images')
    parser.add_argument('--labels-dir', default='data/labels')
    parser.add_argument('--out-dir', default='artifacts/preparation')
    parser.add_argument('--threads', type=int, default=2)
    args = parser.parse_args()
    torch.set_num_threads(args.threads)
    checks = eda(args.images_dir, args.labels_dir, args.out_dir)
    pipeline = pipeline_check(args.images_dir, args.labels_dir, args.out_dir)
    baseline = asdict(train.Config())
    metadata = {'baseline': baseline, 'python': platform.python_version(), 'torch': torch.__version__,
                'timm': timm.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
                'cuda_available': torch.cuda.is_available()}
    (Path(args.out_dir) / 'baseline.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(json.dumps({'eda': checks, 'pipeline': pipeline}, indent=2))


if __name__ == '__main__':
    main()


#### `download_data.py`

In [ ]:
%%writefile code/download_data.py
"""Download and verify official DeepWeeds images and fixed fold-0 labels."""
import hashlib
from pathlib import Path
import shutil
import time
import urllib.request
import zipfile

MD5 = 'b7b30f96d466fba86016aa5a26606e0f'
IMAGE_URL = 'https://zenodo.org/records/7939060/files/images.zip?download=1'
LABEL_BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'


def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'DeepWeeds-Lab/1.0'})
            with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as file:
                shutil.copyfileobj(response, file, length=1024*1024)
            temporary.replace(destination)
            return
        except (OSError, urllib.error.URLError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)


def main():
    data = Path('data')
    archive = data / 'images.zip'
    if not archive.exists():
        print('Downloading official images (~490 MB)...', flush=True)
        download(IMAGE_URL, archive)
    checksum = hashlib.md5()
    with archive.open('rb') as file:
        for chunk in iter(lambda: file.read(1024*1024), b''):
            checksum.update(chunk)
    if checksum.hexdigest() != MD5:
        raise ValueError('Image archive MD5 differs from the lab specification; remove the bad archive and retry.')
    images = data / 'images'
    images.mkdir(parents=True, exist_ok=True)
    seen = set()
    with zipfile.ZipFile(archive) as zip_file:
        for info in zip_file.infolist():
            if info.is_dir() or not info.filename.lower().endswith('.jpg'):
                continue
            name = Path(info.filename).name
            if name in seen:
                raise ValueError(f'Duplicate image basename in archive: {name}')
            seen.add(name)
            target = images / name
            if not target.exists() or target.stat().st_size != info.file_size:
                temporary = target.with_suffix('.part')
                with zip_file.open(info) as source, temporary.open('wb') as dest:
                    shutil.copyfileobj(source, dest)
                temporary.replace(target)
    if len(seen) != 17509:
        raise ValueError(f'Expected 17509 JPEGs, found {len(seen)}')
    for name in ('labels', 'train_subset0', 'val_subset0', 'test_subset0'):
        destination = data / 'labels' / f'{name}.csv'
        if not destination.exists():
            download(f'{LABEL_BASE}/{name}.csv', destination)
    print(f'MD5 OK; {len(seen)} images and original fold-0 CSVs ready.', flush=True)


if __name__ == '__main__':
    main()


#### `experiments.py`

In [ ]:
%%writefile code/experiments.py
"""Run GUIDE steps 1-5. Select on validation; seal all choices before test."""
from __future__ import annotations
import argparse
import hashlib
from dataclasses import asdict, replace
import json
from pathlib import Path
import subprocess
import sys
import numpy as np
import torch
from timm.data import resolve_model_data_config
import dataset
import model as models
import train
import inference as inf
from benchmark import bench
from eval import compute_metrics, save_predictions

STUDENT_ID = '2A202602853'
STUDENT_NAME = 'Hoàng Văn Nam'
DEFAULT_OUTPUT = 'submissions/2A202602853_Hoang_Van_Nam'
BACKBONES = ['resnet50', 'resnext50', 'convnext_tiny', 'deit_small', 'mobilenetv3']
ABLATIONS = [
    ('A', 'scratch', {'init': 'scratch'}), ('A', 'frozen', {'init': 'frozen'}),
    ('B', 'color', {'aug': 'color'}), ('B', 'randaug', {'aug': 'randaug'}),
    ('B', 'mixup', {'mix': 'mixup'}), ('B', 'cutmix', {'mix': 'cutmix'}),
    ('C', 'smoothing', {'loss': 'ls', 'label_smoothing': .1}),
    ('C', 'focal', {'loss': 'focal'}), ('C', 'weighted', {'loss': 'ce_weighted'}),
    ('D', 'balanced', {'sampler': 'balanced'}),
    ('E', 'same_lr', {'lr_backbone': 1e-3}),
    ('F', 'ema', {'ema_decay': .999}),
]
METHODS = ['single', 'hflip_prob', 'five_crop', 'hflip_logit', 'ensemble', 'fused', 'resize256', 'resize288', 'amp']


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def select_best(rows):
    if not rows:
        raise ValueError('No validation results')
    return max(rows, key=lambda r: r['macro_f1_val'])


def base_config(output, epochs=12, batch_size=32, workers=2):
    return train.Config(epochs=epochs, batch_size=batch_size, num_workers=workers,
                        out_dir=str(Path(output) / 'runs'), pred_dir=str(Path(output) / 'predictions'),
                        curves_dir=str(Path(output) / 'curves'), resume=True)


def train_record(cfg, axis='', changes=None):
    summary = train.run(cfg)
    metadata = json.loads((train.run_dir(cfg) / 'config.json').read_text(encoding='utf-8'))
    return {'exp_id': cfg.exp_id, 'seed': cfg.seed, 'backbone': cfg.backbone,
            'axis': axis, 'changes': changes or {}, 'config': asdict(cfg),
            'weight_tag': metadata['pretrained_cfg'].get('hf_hub_id', metadata['pretrained_cfg'].get('url', 'scratch'))
                          if cfg.init != 'scratch' else 'scratch', **summary}


def load_network(cfg, device):
    model = models.build_model(cfg.backbone, pretrained=False, init='scratch', drop_rate=cfg.drop_rate).to(device).eval()
    model.load_state_dict(torch.load(train.run_dir(cfg) / 'best.pt', map_location=device, weights_only=True)['model'])
    meta = json.loads((train.run_dir(cfg) / 'config.json').read_text(encoding='utf-8'))
    return model, meta['preprocessing']


def method_views(x, method, size):
    if method == 'five_crop':
        return inf.views_multicrop(x, size)
    if method.startswith('hflip'):
        return [x, inf.view_hflip(x)]
    return [x]


def prediction(cfg, split, method, partner=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, prep = load_network(cfg, device)
    other = load_network(partner, device)[0] if method == 'ensemble' else None
    if method == 'fused':
        fused = inf.fuse_conv_bn(model).to(device)
        x = torch.randn(2, 3, cfg.img_size, cfg.img_size, device=device)
        with torch.inference_mode():
            torch.testing.assert_close(model(x), fused(x), atol=1e-4, rtol=1e-4)
        model = fused
    size = round(cfg.img_size * 256 / 224) if method == 'five_crop' else 256 if method == 'resize256' else 288 if method == 'resize288' else cfg.img_size
    frame = dataset.load_split(cfg.labels_dir)[1 if split == 'val' else 2]
    loader = dataset.make_loader(frame, cfg.images_dir,
                dataset.build_transforms(False, size, mean=prep['mean'], std=prep['std']),
                cfg.batch_size, False, num_workers=cfg.num_workers)
    names, truths, probabilities = [], [], []
    with torch.inference_mode():
        for x, y, filenames in loader:
            x = x.to(device)
            with torch.autocast(device_type=device.type, enabled=method == 'amp' and device.type == 'cuda'):
                logits = [model(v).float().cpu().numpy() for v in method_views(x, method, cfg.img_size)]
            p = inf.aggregate_views(logits, 'logit' if method == 'hflip_logit' else 'prob')
            if other is not None:
                p = inf.ensemble_probs([p, inf.apply_temperature(other(x).float().cpu().numpy(), 1.)])
            names.extend(filenames)
            truths.append(y.numpy())
            probabilities.append(p)
    return names, np.concatenate(truths), np.concatenate(probabilities)


def method_latency(cfg, method, batch, partner=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, _ = load_network(cfg, device)
    other = load_network(partner, device)[0] if method == 'ensemble' else None
    if method == 'fused':
        model = inf.fuse_conv_bn(model).to(device)
    size = round(cfg.img_size * 256 / 224) if method == 'five_crop' else 256 if method == 'resize256' else 288 if method == 'resize288' else cfg.img_size
    x = torch.randn(batch, 3, size, size, device=device)
    def forward():
        with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=method == 'amp' and device.type == 'cuda'):
            values = [model(v).softmax(1) if method != 'hflip_logit' else model(v)
                      for v in method_views(x, method, cfg.img_size)]
            output = torch.stack(values).mean(0)
            if method == 'hflip_logit':
                output = output.softmax(1)
            if other is not None:
                output = (output + other(x).softmax(1)) / 2
            return output
    timing = bench(forward, iters=50, sync=(lambda: torch.cuda.synchronize(device)) if device.type == 'cuda' else None)
    timing.update(batch=batch, img_size=size, gpu=torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU',
                  dtype='amp' if method == 'amp' else 'fp32', bn_fused=method == 'fused', preprocessing=False,
                  images_per_s=batch / (timing['p50'] / 1000), torch=str(torch.__version__),
                  method=method, checkpoint=str(train.run_dir(cfg) / 'best.pt'))
    return timing


def scalar_metrics(y, p):
    values = compute_metrics(y, p.argmax(1), p)
    return {k: values[k] for k in ('macro_f1', 'top1', 'ece', 'balanced_acc', 'nll')}


def run_backbones(state, base, persist):
    for i, name in enumerate(BACKBONES, 1):
        exp = f'B{i:02d}_{name}'
        if any(r['exp_id'] == exp for r in state['Backbones']):
            continue
        cfg = replace(base, exp_id=exp, backbone=name)
        record = train_record(cfg)
        record.update(latency_ms=method_latency(cfg, 'single', 1)['p50'])
        state['Backbones'].append(record)
        persist()
    return replace(base, backbone=select_best(state['Backbones'])['backbone'])


def run_training(state, base, persist):
    cases = [('T00', '', 'baseline', {})] + [(f'T{i:02d}_{name}', axis, name, changes)
                  for i, (axis, name, changes) in enumerate(ABLATIONS, 1)]
    if 'deit' not in base.backbone and 'swin' not in base.backbone and 'vit' not in base.backbone:
        cases += [('T13_resolution', 'G', 'resolution', {'img_size': 256})]
    for exp, axis, name, changes in cases:
        if any(r['exp_id'] == exp for r in state['Training']):
            continue
        state['Training'].append(train_record(replace(base, exp_id=exp, **changes), axis, changes))
        persist()
    anchor = next(r for r in state['Training'] if r['exp_id'] == 'T00')
    combined = {}
    for axis in 'ABCDEFG':
        rows = [r for r in state['Training'] if r['axis'] == axis]
        if rows:
            winner = select_best(rows)
            if winner['macro_f1_val'] > anchor['macro_f1_val']:
                combined.update(winner['changes'])
    if not any(r['exp_id'] == 'T99_combined' for r in state['Training']):
        state['Training'].append(train_record(replace(base, exp_id='T99_combined', **combined), 'combined', combined))
        persist()
    return train.Config(**select_best(state['Training'])['config'])


def run_inference(state, best, base, persist):
    for i, method in enumerate(METHODS):
        if any(r['method'] == method for r in state['Inference']):
            continue
        if method == 'amp' and not torch.cuda.is_available():
            continue
        if (method.startswith('resize') or method == 'ensemble') and (('deit' in best.backbone or 'swin' in best.backbone or 'vit' in best.backbone) and best.img_size != 224):
            continue
        if method.startswith('resize') and any(k in best.backbone for k in ('deit', 'swin', 'vit')):
            continue  # Fixed-resolution transformer weights cannot accept arbitrary sizes.
        if method == 'ensemble' and best.img_size != base.img_size:
            continue
        if method == 'fused':
            net, _ = load_network(best, 'cpu')
            if not any(isinstance(m, torch.nn.BatchNorm2d) for m in net.modules()):
                continue
        _, y, p = prediction(best, 'val', method, base)
        m = scalar_metrics(y, p)
        temperature = inf.fit_temperature(np.log(np.clip(p, 1e-12, 1)), y)
        calibrated = inf.apply_temperature(np.log(np.clip(p, 1e-12, 1)), temperature)
        latency = method_latency(best, method, 1, base)
        bulk = method_latency(best, method, 32, base)
        state['Latency'].extend([dict(exp_id=f'I{i:02d}', **latency), dict(exp_id=f'I{i:02d}', **bulk)])
        state['Inference'].append({'exp_id': f'I{i:02d}', 'method': method, 'config': asdict(best),
                                  'macro_f1_val': m['macro_f1'], 'top1_val': m['top1'], 'ece_val': m['ece'],
                                  'ece_calibrated_val': scalar_metrics(y, calibrated)['ece'], 'temperature': temperature,
                                  'checkpoint': str(train.run_dir(best) / 'best.pt'),
                                  'K': 5 if method == 'five_crop' else 2 if method.startswith('hflip') or method == 'ensemble' else 1,
                                  **latency, 'bulk_images_per_s': bulk['images_per_s']})
        persist()
    # Accuracy first; lower latency resolves exact validation ties.
    winner = max(state['Inference'], key=lambda r: (r['macro_f1_val'], -r['p95']))
    return winner['method']


def prepare_final(state, best, base, method, persist):
    if state.get('sealed'):
        return
    for seed in (0, 1, 2):
        for cfg in (replace(base, exp_id='T00', seed=seed), replace(best, exp_id='F01', seed=seed)):
            train.run(cfg)
        final = replace(best, exp_id='F01', seed=seed)
        baseline = replace(base, exp_id='T00', seed=seed)
        names, y, p = prediction(final, 'val', method, baseline)
        temperature = inf.fit_temperature(np.log(np.clip(p, 1e-12, 1)), y)
        save_predictions(Path(base.pred_dir) / f'F01_seed{seed}_val.csv', names, y,
                         inf.apply_temperature(np.log(np.clip(p, 1e-12, 1)), temperature))
        state['calibrations'][str(seed)] = temperature
        persist()
    final_zero, baseline_zero = replace(best, exp_id='F01', seed=0), replace(base, exp_id='T00', seed=0)
    if not any(r['exp_id'] == 'F01' for r in state['Latency']):
        state['Latency'].extend([dict(exp_id='F01', **method_latency(final_zero, method, batch, baseline_zero)) for batch in (1, 32)])
    state['sealed'] = {'final_config': asdict(best), 'baseline_config': asdict(base), 'method': method,
                       'seeds': [0, 1, 2], 'selection_split': 'val'}
    persist()  # Frozen BEFORE looking at any test predictions.


def run_test(state, persist):
    seal = state['sealed']
    base = train.Config(**seal['baseline_config'])
    best = train.Config(**seal['final_config'])
    for seed in seal['seeds']:
        if any(r['seed'] == seed for r in state['Final']):
            continue
        final, baseline = replace(best, exp_id='F01', seed=seed), replace(base, exp_id='T00', seed=seed)
        cache = Path(base.out_dir).parent / 'test_cache' / f'seed{seed}.npz'
        if not cache.exists():
            names, y, p = prediction(final, 'test', seal['method'], baseline)
            bn, by, bp = prediction(baseline, 'test', 'single')
            if names != bn or not np.array_equal(y, by):
                raise ValueError('Final/baseline test order differs')
            cache.parent.mkdir(parents=True, exist_ok=True)
            tmp = cache.with_suffix('.tmp')
            with tmp.open('wb') as f:
                np.savez_compressed(f, names=names, y=y, p=p, baseline=bp)
            tmp.replace(cache)
        values = np.load(cache)
        names, y, p, bp = values['names'].tolist(), values['y'], values['p'], values['baseline']
        calibrated = inf.apply_temperature(np.log(np.clip(p, 1e-12, 1)), state['calibrations'][str(seed)])
        for exp, probs in [('F01', calibrated), ('F01uncal', p), ('T00', bp)]:
            path = Path(base.pred_dir) / f'{exp}_seed{seed}_test.csv'
            if not path.exists():
                save_predictions(path, names, y, probs)
        state['Final'].append({'exp_id': 'F01', 'seed': seed, 'configuration': seal,
                               'temperature': state['calibrations'][str(seed)],
                               **{k + '_test': v for k, v in scalar_metrics(y, calibrated).items()},
                               'baseline': scalar_metrics(y, bp)})
        persist()
    output = Path(base.out_dir).parent
    common = ['--test-csv', str(Path(base.labels_dir) / 'test_subset0.csv'), '--labels', str(Path(base.labels_dir) / 'labels.csv')]
    for tag in ('F01', 'T00'):
        subprocess.run([sys.executable, 'eval.py', 'score', '--pred', str(Path(base.pred_dir) / f'{tag}_seed*_test.csv'),
                        *common, '--tag', tag, '--out', str(output / 'eval_out')], check=True)
    subprocess.run([sys.executable, 'eval.py', 'grade', '--final', str(Path(base.pred_dir) / 'F01_seed*_test.csv'),
                    '--baseline', str(Path(base.pred_dir) / 'T00_seed*_test.csv'),
                    '--uncal', str(Path(base.pred_dir) / 'F01uncal_seed*_test.csv'),
                    '--final-val', str(Path(base.pred_dir) / 'F01_seed*_val.csv'),
                    '--val-csv', str(Path(base.labels_dir) / 'val_subset0.csv'),
                    '--latency-p95-ms', str(next(r['p95'] for r in state['Latency'] if r['exp_id'] == 'F01' and r['batch'] == 1)),
                    '--latency-method', 'proper',
                    *common, '--out', str(output / 'eval_out')], check=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--stage', choices=['all', 'backbones', 'training', 'inference', 'final', 'export'], default='all')
    parser.add_argument('--output', default=DEFAULT_OUTPUT)
    parser.add_argument('--epochs', type=int, default=12)
    parser.add_argument('--batch-size', type=int, default=32)
    parser.add_argument('--workers', type=int, default=2)
    parser.add_argument('--allow-cpu', action='store_true')
    args = parser.parse_args()
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=True)
    path = output / 'experiment_state.json'
    state = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {
        'student_id': STUDENT_ID, 'student_name': STUDENT_NAME,
        **{k: [] for k in ['Backbones', 'Training', 'Inference', 'Final', 'Latency']}, 'calibrations': {}}
    def persist():
        write_json(path, state)
    base = base_config(output, args.epochs, args.batch_size, args.workers)
    settings = {'epochs': args.epochs, 'batch_size': args.batch_size, 'workers': args.workers}
    if args.stage != 'export' and 'run_settings' in state and state['run_settings'] != settings:
        raise SystemExit('Settings differ from this saved campaign; restore settings or choose a new output folder.')
    if args.stage != 'export':
        state['run_settings'] = settings
        digest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob('*.py')}
        if 'source_sha256' in state and state['source_sha256'] != digest:
            raise SystemExit('Code differs from this saved campaign. Use original code or a new output folder.')
        state['source_sha256'] = digest
    if args.stage == 'export':
        persist()
        from deliverables import export
        export(output, state)
        return
    if not torch.cuda.is_available() and not args.allow_cpu:
        persist()
        raise SystemExit('No CUDA GPU. Use Colab/Kaggle GPU, or explicitly --allow-cpu for the long CPU run.')
    if args.epochs < 10:
        raise SystemExit('Official experiments require >=10 epochs.')
    if state.get('sealed') and args.stage not in ('all', 'final'):
        raise SystemExit('Test selection is sealed. Screening is closed for this output directory.')
    persist()
    if not state.get('sealed'):
        base = run_backbones(state, base, persist)
        if args.stage == 'backbones':
            return
        best = run_training(state, base, persist)
        if args.stage == 'training':
            return
        method = run_inference(state, best, replace(base, exp_id='T00'), persist)
        if args.stage == 'inference':
            return
        prepare_final(state, best, base, method, persist)
    run_test(state, persist)
    from deliverables import export
    export(output, state)


if __name__ == '__main__':
    main()


#### `deliverables.py`

In [ ]:
%%writefile code/deliverables.py
"""Export the required workbook, measured plots, report and submission bundle."""
from __future__ import annotations
import json
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from eval import read_pred, compute_metrics
import dataset


def table(rows, columns):
    return pd.DataFrame(rows).reindex(columns=columns)


def text_table(frame):
    if frame.empty:
        return 'Chưa có kết quả đo cho phần này.\n'
    def cell(value):
        if isinstance(value, (float, np.floating)):
            return 'n.a.' if pd.isna(value) else f'{value:.4f}'
        return str(value).replace('|', '/')
    lines = ['| ' + ' | '.join(frame.columns) + ' |', '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |']
    lines += ['| ' + ' | '.join(cell(v) for v in row) + ' |' for row in frame.itertuples(index=False, name=None)]
    return '\n'.join(lines) + '\n'


def final_tables(output, state):
    final_rows, per_class = [], []
    for tag in ('T00', 'F01'):
        predictions = sorted((output / 'predictions').glob(f'{tag}_seed*_test.csv'))
        measured = []
        for path in predictions:
            pred = read_pred(str(path))
            metrics = compute_metrics(pred.y_true, pred.y_pred, pred.probs)
            val_path = output / 'predictions' / f'{tag}_seed{pred.seed}_val.csv'
            val = read_pred(str(val_path))
            vm = compute_metrics(val.y_true, val.y_pred, val.probs)
            measured.append(metrics)
            final_rows.append({'exp_id': tag, 'seed': pred.seed, 'configuration': json.dumps(state.get('sealed', {}), ensure_ascii=False),
                               'macro_f1_val': vm['macro_f1'], 'macro_f1_test': metrics['macro_f1'],
                               'top1_test': metrics['top1'], 'ece_test': metrics['ece']})
        if measured:
            aggregate = {'exp_id': tag, 'seed': 'mean ± std (ddof=1)'}
            for key in ('macro_f1', 'top1', 'ece'):
                values = [m[key] for m in measured]
                sd = np.std(values, ddof=1) if len(values) >= 2 else float('nan')
                aggregate[key + '_test'] = f'{np.mean(values):.4f} ± {sd:.4f}'
            final_rows.append(aggregate)
            for label, name in enumerate(dataset.CLASS_NAMES):
                row = {'exp_id': tag, 'class': name, 'test_images_per_seed': int(measured[0]['support'][label])}
                for key in ('precision', 'recall', 'f1'):
                    values = [m[key][label] for m in measured]
                    row[key] = float(np.mean(values))
                    row[key + '_std'] = float(np.std(values, ddof=1)) if len(values) > 1 else None
                per_class.append(row)
            cm = sum(m['confusion'] for m in measured)
            fig, ax = plt.subplots(figsize=(10, 9))
            plot = ax.imshow(cm, cmap='Blues')
            ax.set(xticks=range(9), yticks=range(9), xticklabels=dataset.CLASS_NAMES,
                   yticklabels=dataset.CLASS_NAMES, xlabel='Predicted', ylabel='True', title=f'{tag}: summed test confusion, {len(measured)} seeds')
            plt.setp(ax.get_xticklabels(), rotation=65, ha='right')
            for i in range(9):
                for j in range(9):
                    ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=7,
                            color='white' if cm[i, j] > cm.max()/2 else 'black')
            fig.colorbar(plot, ax=ax)
            fig.tight_layout()
            fig.savefig(output / 'curves' / f'{tag}_confusion.png', dpi=150)
            plt.close(fig)
    return table(final_rows, ['exp_id', 'configuration', 'seed', 'macro_f1_val', 'macro_f1_test', 'top1_test', 'ece_test']), table(per_class,
        ['exp_id', 'class', 'test_images_per_seed', 'precision', 'precision_std', 'recall', 'recall_std', 'f1', 'f1_std'])


def error_gallery(output, state):
    path = output / 'predictions' / 'F01_seed0_test.csv'
    if not path.exists():
        return []
    pred = read_pred(str(path))
    wrong = np.flatnonzero(pred.y_true != pred.y_pred)
    indices = sorted(wrong, key=lambda i: -pred.probs[i].max())[:9]
    if not indices:
        return []
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    cfg = state['sealed']['final_config']
    rows = []
    for ax in axes.flat:
        ax.axis('off')
    for ax, i in zip(axes.flat, indices):
        filename = pred.filenames[i]
        with Image.open(Path(cfg['images_dir']) / filename) as image:
            ax.imshow(image.convert('RGB'))
        true, guess = dataset.CLASS_NAMES[pred.y_true[i]], dataset.CLASS_NAMES[pred.y_pred[i]]
        ax.set_title(f'{filename}\nTrue: {true}\nPred: {guess}; confidence={pred.probs[i].max():.3f}', fontsize=9)
        rows.append({'filename': filename, 'true': true, 'predicted': guess, 'confidence': float(pred.probs[i].max())})
    fig.tight_layout()
    fig.savefig(output / 'curves' / 'F01_errors.png', dpi=150)
    plt.close(fig)
    pd.DataFrame(rows).to_csv(output / 'eval_out' / 'F01_errors.csv', index=False)
    return rows


def export(output, state):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / 'curves').mkdir(exist_ok=True)
    (output / 'eval_out').mkdir(exist_ok=True)
    backbone_rows = []
    for r in state['Backbones']:
        c = r['config']
        backbone_rows.append(dict(exp_id=r['exp_id'], backbone=r['backbone'], weight_tag=r['weight_tag'],
            params_m=r['params_m'], gmacs=r['gmacs'], img_size=c['img_size'], epochs=c['epochs'], seed=r['seed'],
            macro_f1_val=r['macro_f1_val'], top1_val=r.get('top1_val'),
            train_seconds_per_epoch=r['mean_epoch_seconds'], batch1_ms=r['latency_ms'], notes=r['gmacs_note']))
    backbones = table(backbone_rows, ['exp_id', 'backbone', 'weight_tag', 'params_m', 'gmacs', 'img_size', 'epochs', 'seed',
                    'macro_f1_val', 'top1_val', 'train_seconds_per_epoch', 'batch1_ms', 'notes'])
    anchor = next((r['macro_f1_val'] for r in state['Training'] if r['exp_id'] == 'T00'), None)
    training_rows = []
    for r in state['Training']:
        pred = read_pred(str(output / 'predictions' / f"{r['exp_id']}_seed{r['seed']}_val.csv"))
        m = compute_metrics(pred.y_true, pred.y_pred, pred.probs)
        training_rows.append(dict(exp_id=r['exp_id'], backbone=r['backbone'], axis=r['axis'],
            changes=json.dumps(r['changes'], ensure_ascii=False), seed=r['seed'], macro_f1_val=r['macro_f1_val'],
            top1_val=m['top1'], delta_vs_T00=r['macro_f1_val'] - anchor,
            f1_chinee_apple=m['f1'][0], f1_snake_weed=m['f1'][7],
            notes='One screening seed; differences do not establish statistical significance'))
    training = table(training_rows, ['exp_id', 'backbone', 'axis', 'changes', 'seed', 'macro_f1_val', 'top1_val',
                                   'delta_vs_T00', 'f1_chinee_apple', 'f1_snake_weed', 'notes'])
    inference_rows = [dict(r) for r in state['Inference']]
    reference = next((r['p50'] for r in inference_rows if r['method'] == 'single'), None)
    for r in inference_rows:
        r['relative_cost'] = r['p50'] / reference
    inference = table(inference_rows, ['exp_id', 'method', 'checkpoint', 'K', 'macro_f1_val', 'top1_val', 'ece_val',
                      'temperature', 'ece_calibrated_val', 'p50', 'p95', 'p99', 'bulk_images_per_s', 'relative_cost'])
    final, per_class = final_tables(output, state)
    latency = table(state['Latency'], ['exp_id', 'method', 'checkpoint', 'gpu', 'dtype', 'batch', 'img_size', 'bn_fused',
                                      'p50', 'p95', 'p99', 'images_per_s', 'preprocessing', 'torch'])
    ranking = [{'exp_id': r['exp_id'], 'stage': stage, 'backbone': r['backbone'], 'macro_f1_val': r['macro_f1_val'],
                'batch1_p50_ms': r.get('latency_ms'), 'seed': r['seed']} for stage in ('Backbones', 'Training') for r in state[stage]]
    ranking += [{'exp_id': r['exp_id'], 'stage': 'Inference', 'backbone': r['config']['backbone'],
                 'macro_f1_val': r['macro_f1_val'], 'batch1_p50_ms': r['p50'], 'seed': 0} for r in state['Inference']]
    summary = table(sorted(ranking, key=lambda r: -r['macro_f1_val'])[:10],
                    ['exp_id', 'stage', 'backbone', 'macro_f1_val', 'batch1_p50_ms', 'seed'])
    sheets = {'Summary': summary, 'Backbones': backbones, 'Training': training, 'Inference': inference,
              'Final': final, 'PerClass': per_class, 'Latency': latency}
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    with pd.ExcelWriter(output / 'results.xlsx', engine='openpyxl') as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name, index=False, startrow=3)
            ws = writer.sheets[name]
            ws['A1'] = f"{state['student_id']} — {state['student_name']} — {name}"
            ws['A1'].font = Font(bold=True, size=14)
            ws['A2'] = 'Source: experiment_state.json, runs/*/history.csv and original eval.py; unavailable values remain blank.'
            ws.freeze_panes = 'C5'
            ws.auto_filter.ref = f'A4:{get_column_letter(len(frame.columns))}{max(4, len(frame)+4)}'
            ws.sheet_view.showGridLines = False
            for cell in ws[4]:
                cell.font = Font(color='FFFFFF', bold=True)
                cell.fill = PatternFill('solid', fgColor='17365D')
                cell.alignment = Alignment(wrap_text=True, vertical='center')
            ws.row_dimensions[4].height = 42
            for col, label in enumerate(frame.columns, 1):
                ws.column_dimensions[get_column_letter(col)].width = min(36, max(15, len(label)+3))
            for row in ws.iter_rows(min_row=5):
                for cell in row:
                    cell.alignment = Alignment(wrap_text=True, vertical='top')
                    if isinstance(cell.value, float):
                        cell.number_format = '0.0000'
            if frame.empty:
                ws['A5'] = 'NOT RUN — no measured results for this stage yet.'
                ws.merge_cells(start_row=5, start_column=1, end_row=5, end_column=len(frame.columns))
            ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(frame.columns))
            ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=len(frame.columns))
            ws['A2'].alignment = Alignment(wrap_text=True)
            ws.row_dimensions[2].height = 32
            if 'macro_f1_val' in frame and not frame.empty:
                row_number = int(frame['macro_f1_val'].astype(float).idxmax()) + 5
                for cell in ws[row_number]:
                    cell.fill = PatternFill('solid', fgColor='E2F0D9')
    if not inference.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(inference.p95, inference.macro_f1_val)
        for r in inference.itertuples():
            ax.annotate(r.method, (r.p95, r.macro_f1_val), fontsize=8)
        ax.set(xlabel='p95 latency (ms, batch 1)', ylabel='Validation macro-F1', title='Accuracy / latency trade-off')
        fig.tight_layout()
        fig.savefig(output / 'curves' / 'inference_tradeoff.png', dpi=150)
        plt.close(fig)
    if not backbones.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(backbones.batch1_ms, backbones.macro_f1_val)
        for r in backbones.itertuples():
            ax.annotate(r.backbone, (r.batch1_ms, r.macro_f1_val), fontsize=8)
        ax.set(xlabel='Latency (ms, batch 1)', ylabel='Validation macro-F1', title='Backbone screening')
        fig.tight_layout()
        fig.savefig(output / 'curves' / 'backbone_tradeoff.png', dpi=150)
        plt.close(fig)
    errors = error_gallery(output, state)
    root = Path(__file__).resolve().parents[1]
    preparation = root / 'artifacts' / 'preparation'
    if preparation.exists():
        shutil.copytree(preparation, output / 'artifacts' / 'preparation', dirs_exist_ok=True)
    if (root / 'report.md').exists():
        prep_report = (root / 'report.md').read_text(encoding='utf-8')
    else:
        prep_report = (preparation / 'eda.md').read_text(encoding='utf-8') if (preparation / 'eda.md').exists() else 'Run preparation first.'
        prep_report = prep_report.replace('(class_', '(artifacts/preparation/class_')
    lines = [f"# Báo cáo Lab Day 2 — {state['student_name']}", f"Mã học viên: **{state['student_id']}**", '',
        '## 1. Tóm tắt',
        f"Đã ghi nhận {len(backbones)} backbone, {len(training)} công thức huấn luyện, {len(inference)} phương pháp suy luận và {len(state['Final'])} seed chung kết.",
        'Các bảng dưới đây chỉ lấy từ dữ liệu đã đo; phần chưa chạy không có số liệu.', '',
        '## 2. Dữ liệu và thiết lập', prep_report, '',
        'Các thí nghiệm chính dùng fold 0 cố định, chọn checkpoint bằng macro-F1 validation. '
        'T00 là công thức nền trên backbone được chọn từ Bước 1; B01_resnet50 giữ mốc kiến trúc ResNet-50. '
        'Batch mặc định 32 để phù hợp GPU T4 (giảm từ 64 gợi ý trong GUIDE); 12 epoch; AdamW, warmup 1 epoch, cosine; AMP train trên CUDA. '
        'Nhãn train không sửa theo labels.csv khi có sai lệch upstream đã ghi trong EDA.', '',
        '## 3. So sánh backbone', text_table(backbones[['backbone', 'macro_f1_val', 'top1_val', 'params_m', 'gmacs', 'batch1_ms']]),
        'Backbone có macro-F1 val cao nhất được chọn. Đây là sàng lọc một seed; trọng số tiền huấn luyện '
        'có thể có công thức khác nhau nên không quy mọi chênh lệch cho riêng kiến trúc. GMAC bị để trống nếu fvcore gặp phép toán chưa hỗ trợ.', '',
        '## 4. Ablation công thức', text_table(training[['exp_id', 'axis', 'changes', 'macro_f1_val', 'delta_vs_T00']]),
        'Mỗi dòng A–G thay một yếu tố; T99 kết hợp các thay đổi có delta dương trên từng trục. '
        'Kết hợp không mặc nhiên tốt hơn: vẫn được đo lại và so với tất cả công thức. '
        'Không kết luận ý nghĩa thống kê từ một seed sàng lọc. Label smoothing giảm độ tự tin; focal ưu tiên mẫu khó; '
        'loss có trọng số và sampler tác động khác nhau lên lớp hiếm. Đây là cơ chế lý thuyết, hiệu quả thực tế phải đọc từ bảng.', '',
        '## 5. Phương pháp suy luận', text_table(inference[['method', 'macro_f1_val', 'ece_val', 'ece_calibrated_val', 'p95', 'relative_cost']]),
        '![Đánh đổi](curves/inference_tradeoff.png)' if not inference.empty else 'Chưa có biểu đồ suy luận.',
        'Latency đo 10 warmup + 50 lượt, đồng bộ CUDA trước/sau, batch 1 và 32. Không tính decode ảnh/tiền xử lý PIL hoặc I/O. '
        'Temperature khớp bằng NLL trên validation, bảo toàn argmax; giảm NLL không bảo đảm ECE cũng giảm. '
        'ViT/Swin có kích thước cố định không chạy resize tùy ý. BN fusion chỉ áp dụng kiến trúc có BatchNorm và kiểm tra tương đương.', '',
        '## 6. Chung kết và test', text_table(final),
        'Ba seed 0/1/2 cho cả F01 và T00; mean ± sample std (ddof=1). Mọi lựa chọn được khóa trước lần test đầu tiên. '
        'Test cache cho phép xuất lại báo cáo mà không chạy lại model trên test. Không dùng test để chọn hoặc sửa công thức.', '',
        '## 7. Phân tích lỗi và khuyến nghị']
    if state['Final']:
        fv = np.array([r['macro_f1_test'] for r in state['Final']])
        bv = np.array([r['baseline']['macro_f1'] for r in state['Final']])
        noise = max(np.std(fv, ddof=1), np.std(bv, ddof=1))
        lines += [f'Macro-F1: F01 {fv.mean():.4f} ± {fv.std(ddof=1):.4f}; T00 {bv.mean():.4f} ± {bv.std(ddof=1):.4f}; delta {fv.mean()-bv.mean():+.4f}.',
                  'Chênh lệch lớn hơn std đo được; vẫn chưa phải kiểm định ý nghĩa thống kê.' if abs(fv.mean()-bv.mean()) > noise else 'Chênh lệch không vượt std: chưa phân biệt được hai cấu hình.',
                  '![Ma trận nhầm lẫn](curves/F01_confusion.png)']
        if errors:
            lines += ['![Ảnh dự đoán sai, seed 0](curves/F01_errors.png)', text_table(pd.DataFrame(errors)),
                      'Các cặp nhầm lẫn và ảnh trên là quan sát thực tế. Nền lá/cỏ tương tự, che khuất và ánh sáng là giả thuyết cần kiểm tra bằng mắt; không tự động khẳng định nguyên nhân sinh học.']
    else:
        lines += ['Chưa chạy chung kết; chưa có kết luận về chất lượng test.']
    if not inference.empty:
        for budget in (30, 100):
            eligible = inference[inference.p95 <= budget]
            lines += [f'Ngân sách {budget} ms: ' + (f"chọn {eligible.sort_values('macro_f1_val', ascending=False).iloc[0]['method']} theo macro-F1 validation trong các cấu hình đáp ứng p95." if not eligible.empty else 'không có cấu hình đo được đáp ứng trên phần cứng này.')]
    lines += ['', '## 8. Hạn chế',
              'Một fold, ba seed vòng cuối và một seed sàng lọc. Split ngẫu nhiên theo ảnh có thể lạc quan khi chuyển địa điểm/mùa. '
              'Chưa đo end-to-end trên robot; độ trễ Colab không thay thế latency thiết bị triển khai. '
              'Calibration dùng cùng val chọn công thức, có thể lạc quan; miền mới cần đánh giá riêng. '
              'Không đối chiếu trực tiếp số của bài báo 100 epoch với thiết lập 12 epoch này.', '',
              '## 9. Phụ lục tái lập',
              'Cấu hình, tag trọng số, phiên bản thư viện và phần cứng: `runs/*/seed*/config.json`. '
              'Danh sách công thức và lựa chọn khóa: `experiment_state.json`. Notebook tự chứa toàn bộ code: `code/lab_day2.ipynb`. '
              'Kết quả chấm gốc: `eval_out/`; các file dự đoán có p0…p8, argmax và nhãn thật.']
    (output / 'report.md').write_text('\n\n'.join(lines) + '\n', encoding='utf-8')
    source = Path(__file__).resolve().parent
    target = output / 'code'
    if source.resolve() != target.resolve():
        shutil.copytree(source, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    shutil.copy2(root / 'eval.py', output / 'eval.py')
    for name in ('dochieu.md',):
        if (root / name).exists():
            shutil.copy2(root / name, output / name)
    readme = f'''# Lab Day 2 — {state['student_name']} ({state['student_id']})

Mở `code/lab_day2.ipynb` trên Google Colab, chọn GPU rồi Run all.
Notebook tự chứa code, tải DeepWeeds kiểm MD5, lưu Drive và resume theo epoch.
Nếu dừng phiên, mở lại notebook, giữ nguyên cấu hình và chạy lại; không đổi thư mục output sau khi đã mở test.

Các bước: chuẩn bị → backbone → training → inference → final/test → export.
Seed sàng lọc 0; vòng cuối 0,1,2. T00 dùng công thức nền trên backbone được chọn; B01 giữ mốc ResNet-50.
Cấu hình chính: {state.get('run_settings', {})}. Chọn hoàn toàn bằng validation.

`results.xlsx` có 7 sheet yêu cầu. `report.md` được sinh từ số liệu đo.
`runs/` chứa log/cấu hình/checkpoint; không đưa checkpoint hoặc dataset vào ZIP bài nộp.
`eval_out/` chứa kết quả `eval.py score/grade`. Trọng số/thư viện/phần cứng thực tế có trong từng config.json.
`dochieu.md` giải thích từng file.

Notebook chưa có URL chia sẻ công khai: sau khi lưu vào Colab, dùng Share để lấy link của bạn.
'''
    (output / 'README.md').write_text(readme, encoding='utf-8')
    print(f'Exported {output / "results.xlsx"} and report.md from measured data.', flush=True)


#### `requirements.txt`

In [ ]:
%%writefile code/requirements.txt
torch==2.8.0
torchvision==0.23.0
timm==1.0.30
numpy>=1.26,<3
pandas>=2,<3
matplotlib>=3.8,<4
Pillow>=10,<13
scikit-learn>=1.4,<2
scipy>=1.11,<2
openpyxl>=3.1,<4
fvcore==0.1.5.post20221221


#### eval.py gốc — giữ nguyên

In [ ]:
%%writefile eval.py
#!/usr/bin/env python3
"""eval.py - công cụ đánh giá chính thức của Lab Day 2 (DeepWeeds).

File này đã hoàn chỉnh: sinh viên KHÔNG sửa công thức chỉ số. Mọi con số trong
results.xlsx và báo cáo phải khớp với kết quả của file này (RUBRIC.md, mục I).
Chỉ phụ thuộc numpy và pandas.

Định nghĩa chỉ số (README.md, mục 2.2)
--------------------------------------
- top-1 accuracy : số ảnh đoán đúng / tổng số ảnh, không trọng số theo lớp.
- macro-F1       : trung bình cộng F1 của 9 lớp (lớp không có dự đoán đúng có F1 = 0).
- balanced acc.  : trung bình recall của các lớp có ảnh trong tập.
- precision / recall / F1 theo lớp, ma trận nhầm lẫn (hàng = nhãn thật).
- ECE            : 15 bin đều theo độ tin cậy (= max softmax); bin m là (b_{m-1}, b_m].
                   ECE = sum_m (n_m / n) * |acc_m - conf_m|
- mean +- std    : qua các seed, std mẫu (ddof = 1).

Định dạng file dự đoán (mỗi seed một file)
------------------------------------------
    Filename, y_true, y_pred, p0, p1, ..., p8
p0..p8 là xác suất softmax, thứ tự theo cột `Label` của labels.csv
(0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives). Tên file chứa `seed<k>`,
ví dụ predictions/F01_seed0_test.csv. Hàm `save_predictions(...)` trong file này ghi
đúng định dạng đó (`from eval import save_predictions`).

Cách dùng
---------
    # 1) Tính chỉ số cho một nhóm file (một cấu hình, nhiều seed)
    python eval.py score --pred "predictions/F01_seed*_test.csv" \
        --test-csv labels/test_subset0.csv --labels labels/labels.csv --tag F01

    # 2) Tự chấm phần I của RUBRIC (chung kết so với mốc)
    python eval.py grade \
        --final    "predictions/F01_seed*_test.csv" \
        --baseline "predictions/T00_seed*_test.csv" \
        --uncal    "predictions/F01_uncal_seed*_test.csv" \
        --final-val "predictions/F01_seed*_val.csv" \
        --latency-p95-ms 41.5 --latency-method proper \
        --test-csv labels/test_subset0.csv --labels labels/labels.csv

`--uncal` (cùng cấu hình nhưng chưa temperature scaling) và `--final-val` (dự đoán
trên val) là tùy chọn; thiếu thì các ý I4(a) hoặc I4(b) ghi là "chưa chấm được".
"""
from __future__ import annotations

import argparse
import glob
import json
import math
import re
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------- #
# Hằng số
# --------------------------------------------------------------------------- #
NUM_CLASSES = 9
ECE_BINS = 15
PROB_SUM_TOL = 1e-3  # xác suất mỗi dòng phải cộng bằng 1 (sai số cho phép)

# Thứ tự lớp theo cột `Label` của labels.csv (lấy từ deepweeds.py của tác giả).
CLASS_NAMES = [
    "Chinee Apple", "Lantana", "Parkinsonia", "Parthenium", "Prickly Acacia",
    "Rubber Vine", "Siam Weed", "Snake Weed", "Negatives",
]

# Hai lớp khó nhất và mốc recall (%) từ bài báo gốc (README.md, mục 2.3).
HARD_CLASSES = {"Chinee Apple": 88.5, "Snake Weed": 88.8}

# ---- Ngưỡng chấm RUBRIC mục I (TẠM THỜI: giảng viên có thể chỉnh tại đây) ---- #
ACC_TIERS = [(95.7, 7), (95.1, 6), (94.0, 5), (92.0, 3), (90.0, 1)]  # I1: (acc %, điểm)
I2_MIN_DELTA = 0.01          # I2: Delta macro-F1 tối thiểu để đạt 5 điểm
HARD_FLOORS = [(85.0, 3), (80.0, 2)]  # I3: sàn recall (%) cho cả hai lớp khó
I3_MIN_POINTS = 1            # I3: có báo cáo đầy đủ nhưng dưới 80%
I4_MAX_VAL_TEST_GAP = 0.02   # I4(b): |macro-F1 val - macro-F1 test| tối đa
I5_P95_BUDGET_MS = 100.0     # I5: ngân sách p95 ở batch 1
MIN_SEEDS = 3


# --------------------------------------------------------------------------- #
# Đọc và kiểm tra file dự đoán
# --------------------------------------------------------------------------- #
@dataclass
class Pred:
    path: str
    seed: int | None
    filenames: np.ndarray
    y_true: np.ndarray
    y_pred: np.ndarray
    probs: np.ndarray


def parse_seed(path: str) -> int | None:
    m = re.search(r"seed[_-]?(\d+)", Path(path).name)
    return int(m.group(1)) if m else None


def read_pred(path: str, num_classes: int = NUM_CLASSES) -> Pred:
    """Đọc một file dự đoán và kiểm tra định dạng. Ném ValueError nếu sai."""
    df = pd.read_csv(path)
    prob_cols = [f"p{i}" for i in range(num_classes)]
    missing = [c for c in ["Filename", "y_true", "y_pred", *prob_cols] if c not in df.columns]
    if missing:
        raise ValueError(f"{path}: thiếu cột {missing}")
    extra_p = [c for c in df.columns if re.fullmatch(r"p\d+", c) and c not in prob_cols]
    if extra_p:
        raise ValueError(f"{path}: có cột xác suất ngoài p0..p{num_classes - 1}: {extra_p}")
    if len(df) == 0:
        raise ValueError(f"{path}: file rỗng")
    if df["Filename"].duplicated().any():
        raise ValueError(f"{path}: có Filename bị trùng")

    probs = df[prob_cols].to_numpy(dtype=np.float64)
    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1 + 1e-6).any():
        raise ValueError(f"{path}: xác suất chứa NaN, âm hoặc lớn hơn 1; hãy lưu softmax, không lưu logit")
    bad_sum = np.abs(probs.sum(1) - 1.0) > PROB_SUM_TOL
    if bad_sum.any():
        raise ValueError(f"{path}: {int(bad_sum.sum())} dòng có tổng xác suất khác 1 "
                         f"(sai số > {PROB_SUM_TOL}); hãy lưu softmax, không lưu logit")

    ints = {}
    for col in ("y_true", "y_pred"):
        v = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if not np.isfinite(v).all() or (v != np.round(v)).any():
            raise ValueError(f"{path}: cột {col} phải là số nguyên")
        v = v.astype(np.int64)
        if v.min() < 0 or v.max() >= num_classes:
            raise ValueError(f"{path}: cột {col} ngoài khoảng 0..{num_classes - 1}")
        ints[col] = v

    argmax = probs.argmax(1)
    mismatch = int((argmax != ints["y_pred"]).sum())
    if mismatch:
        raise ValueError(f"{path}: {mismatch} dòng có y_pred khác argmax của p0..p{num_classes - 1}")

    return Pred(path, parse_seed(path), df["Filename"].to_numpy(), ints["y_true"], ints["y_pred"], probs)


def save_predictions(path: str | Path, filenames, y_true, probs, num_classes: int = NUM_CLASSES) -> Path:
    """Ghi một file dự đoán đúng định dạng của eval.py: Filename, y_true, y_pred, p0..p{K-1}.

    Dùng hàm này (hoặc ghi y hệt định dạng) để tạo predictions/<exp_id>_seed<k>_<split>.csv.
    `probs` là XÁC SUẤT softmax (mỗi dòng cộng bằng 1), không phải logit.
    y_pred được tính bằng argmax của probs. Kết quả luôn qua được `read_pred`.
    """
    probs = np.asarray(probs, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.int64)
    filenames = list(filenames)
    if probs.ndim != 2 or probs.shape[1] != num_classes:
        raise ValueError(f"probs phải có dạng (N, {num_classes}), nhận {probs.shape}")
    if not (len(filenames) == len(y_true) == len(probs)):
        raise ValueError("filenames, y_true, probs phải cùng độ dài")
    if np.abs(probs.sum(1) - 1.0).max() > PROB_SUM_TOL:
        raise ValueError("probs chưa chuẩn hoá (tổng mỗi dòng khác 1); hãy áp dụng softmax trước khi lưu")
    df = pd.DataFrame({"Filename": filenames, "y_true": y_true, "y_pred": probs.argmax(1)})
    for i in range(num_classes):
        df[f"p{i}"] = probs[:, i]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, float_format="%.8g")
    return path


def check_against_csv(pred: Pred, csv_path: str, what: str = "test") -> None:
    """Đối chiếu tên file và nhãn thật với file CSV chia sẵn (Filename, Label)."""
    ref = pd.read_csv(csv_path)
    if not {"Filename", "Label"} <= set(ref.columns):
        raise ValueError(f"{csv_path}: cần cột Filename và Label")
    ref_map = dict(zip(ref["Filename"], ref["Label"].astype(int)))
    names = set(pred.filenames)
    if names != set(ref_map):
        only_pred = len(names - set(ref_map))
        only_ref = len(set(ref_map) - names)
        raise ValueError(f"{pred.path}: không khớp {what} CSV ({csv_path}): "
                         f"thừa {only_pred} ảnh, thiếu {only_ref} ảnh")
    wrong = sum(int(ref_map[f]) != int(t) for f, t in zip(pred.filenames, pred.y_true))
    if wrong:
        raise ValueError(f"{pred.path}: {wrong} dòng có y_true khác Label trong {csv_path}")


def load_names(labels_csv: str | None) -> list[str]:
    """Lấy tên lớp từ labels.csv (cột Label, Species) nếu có; nếu không dùng mặc định."""
    if not labels_csv:
        return list(CLASS_NAMES)
    df = pd.read_csv(labels_csv)
    if not {"Label", "Species"} <= set(df.columns):
        raise ValueError(f"{labels_csv}: cần cột Label và Species")
    m = df.drop_duplicates("Label").sort_values("Label")
    if list(m["Label"]) != list(range(NUM_CLASSES)):
        raise ValueError(f"{labels_csv}: Label phải là 0..{NUM_CLASSES - 1}")
    return [str(s) for s in m["Species"]]


def _norm(name: str) -> str:
    return re.sub(r"[^a-z]", "", name.lower())


def hard_class_indices(names: list[str]) -> dict[str, int]:
    out = {}
    for hard in HARD_CLASSES:
        for i, n in enumerate(names):
            if _norm(n) == _norm(hard):
                out[hard] = i
    missing = set(HARD_CLASSES) - set(out)
    if missing:
        raise ValueError(f"không tìm thấy lớp {sorted(missing)} trong danh sách tên lớp {names}")
    return out


def expand(pattern: str | list[str]) -> list[str]:
    patterns = [pattern] if isinstance(pattern, str) else list(pattern)
    files: list[str] = []
    for p in patterns:
        files.extend(glob.glob(p))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(f"không có file nào khớp: {patterns}")
    return files


# --------------------------------------------------------------------------- #
# Chỉ số
# --------------------------------------------------------------------------- #
def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int = NUM_CLASSES) -> np.ndarray:
    cm = np.zeros((k, k), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


def per_class(cm: np.ndarray) -> dict[str, np.ndarray]:
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(1).astype(np.float64)
    predicted = cm.sum(0).astype(np.float64)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)
    return {"precision": precision, "recall": recall, "f1": f1, "support": support}


def ece_score(probs: np.ndarray, y_true: np.ndarray, bins: int = ECE_BINS) -> float:
    """ECE với `bins` bin đều; độ tin cậy = max softmax; bin m là (b_{m-1}, b_m]."""
    conf = probs.max(1)
    correct = (probs.argmax(1) == y_true).astype(np.float64)
    idx = np.clip(np.ceil(conf * bins).astype(np.int64) - 1, 0, bins - 1)
    n = len(conf)
    ece = 0.0
    for m in range(bins):
        mask = idx == m
        if mask.any():
            ece += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,
                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:
    cm = confusion_matrix(y_true, y_pred, k)
    pc = per_class(cm)
    has_support = pc["support"] > 0
    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())
    return {
        "n": int(len(y_true)),
        "top1": float((y_true == y_pred).mean()),
        "macro_f1": float(pc["f1"].mean()),
        "balanced_acc": float(pc["recall"][has_support].mean()),
        "ece": ece_score(probs, y_true, bins),
        "nll": nll,
        "precision": pc["precision"], "recall": pc["recall"], "f1": pc["f1"],
        "support": pc["support"], "confusion": cm,
    }


SCALARS = ("top1", "macro_f1", "balanced_acc", "ece", "nll")
VECTORS = ("precision", "recall", "f1")


def mean_std(values) -> tuple[float, float]:
    """mean và std mẫu (ddof=1) theo trục seed; std là NaN nếu chỉ có 1 seed."""
    a = np.asarray(values, dtype=np.float64)
    mean = a.mean(axis=0)
    std = a.std(axis=0, ddof=1) if len(a) > 1 else np.full(a.shape[1:], np.nan)
    if a.ndim == 1:
        return float(mean), float(std)
    return mean, std


@dataclass
class Group:
    preds: list[Pred]
    metrics: list[dict]
    summary: dict  # {"top1": (mean, std), ..., "recall": (vec_mean, vec_std), ...}

    @property
    def seeds(self) -> list[int | None]:
        return [p.seed for p in self.preds]


def load_group(pattern, ref_csv: str | None, k: int = NUM_CLASSES, ref_what: str = "test") -> Group:
    files = expand(pattern)
    preds = [read_pred(f, k) for f in files]
    if ref_csv:
        for p in preds:
            check_against_csv(p, ref_csv, ref_what)
    base_names = set(preds[0].filenames)
    for p in preds[1:]:
        if set(p.filenames) != base_names:
            raise ValueError(f"{p.path}: tập Filename khác {preds[0].path}")
    seeds = [p.seed for p in preds if p.seed is not None]
    if len(seeds) != len(set(seeds)):
        raise ValueError(f"seed bị trùng giữa các file: {files}")
    metrics = [compute_metrics(p.y_true, p.y_pred, p.probs, k) for p in preds]
    summary = {key: mean_std([m[key] for m in metrics]) for key in (*SCALARS, *VECTORS)}
    return Group(preds, metrics, summary)


# --------------------------------------------------------------------------- #
# Định dạng đầu ra
# --------------------------------------------------------------------------- #
def fmt(mean: float, std: float, digits: int = 4) -> str:
    if std is None or (isinstance(std, float) and math.isnan(std)):
        return f"{mean:.{digits}f} (1 seed)"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def report_group(name: str, g: Group, names: list[str]) -> str:
    lines = [f"### {name} ({len(g.preds)} seed: {g.seeds}; {g.metrics[0]['n']} ảnh)", ""]
    if len(g.preds) < MIN_SEEDS:
        lines += [f"> CẢNH BÁO: chỉ có {len(g.preds)} seed, yêu cầu tối thiểu {MIN_SEEDS}.", ""]
    lines += ["| Chỉ số | mean ± std |", "|---|---|"]
    labels = {"top1": "top-1 accuracy", "macro_f1": "macro-F1", "balanced_acc": "balanced accuracy",
              "ece": "ECE (15 bin)", "nll": "NLL"}
    for key in SCALARS:
        lines.append(f"| {labels[key]} | {fmt(*g.summary[key])} |")
    lines += ["", "| Lớp | Precision | Recall | F1 | Số ảnh |", "|---|---|---|---|---|"]
    pm, ps = g.summary["precision"]
    rm, rs = g.summary["recall"]
    fm, fs = g.summary["f1"]
    for i, n in enumerate(names):
        lines.append(f"| {n} | {fmt(pm[i], ps[i], 3)} | {fmt(rm[i], rs[i], 3)} | "
                     f"{fmt(fm[i], fs[i], 3)} | {int(g.metrics[0]['support'][i])} |")
    return "\n".join(lines)


def save_group(out_dir: Path, tag: str, g: Group, names: list[str]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for p, m in zip(g.preds, g.metrics):
        rows.append({"file": Path(p.path).name, "seed": p.seed,
                     **{k: m[k] for k in ("n", *SCALARS)}})
    pd.DataFrame(rows).to_csv(out_dir / f"{tag}_per_seed.csv", index=False)
    pc = pd.DataFrame({"class": names})
    for key in VECTORS:
        mean, std = g.summary[key]
        pc[f"{key}_mean"], pc[f"{key}_std"] = mean, std
    pc["support"] = g.metrics[0]["support"].astype(int)
    pc.to_csv(out_dir / f"{tag}_per_class.csv", index=False)
    cm = sum(m["confusion"] for m in g.metrics)
    pd.DataFrame(cm, index=[f"true_{n}" for n in names],
                 columns=[f"pred_{n}" for n in names]).to_csv(out_dir / f"{tag}_confusion_sum.csv")
    summary = {"seeds": g.seeds,
               **{k: {"mean": g.summary[k][0], "std": g.summary[k][1]} for k in SCALARS},
               **{k: {"mean": g.summary[k][0].tolist(), "std": g.summary[k][1].tolist()} for k in VECTORS},
               "classes": names}
    (out_dir / f"{tag}_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))


# --------------------------------------------------------------------------- #
# Lệnh `score`
# --------------------------------------------------------------------------- #
def cmd_score(args) -> int:
    names = load_names(args.labels)
    g = load_group(args.pred, args.test_csv, ref_what="test")
    if not args.test_csv:
        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\n")
    print(report_group(args.tag, g, names))
    if args.out:
        save_group(Path(args.out), args.tag, g, names)
        print(f"\nĐã lưu kết quả vào {args.out}/{args.tag}_*")
    return 0


# --------------------------------------------------------------------------- #
# Lệnh `grade` (RUBRIC mục I)
# --------------------------------------------------------------------------- #
def points_i1(acc_pct: float) -> int:
    for thr, pts in ACC_TIERS:
        if acc_pct >= thr:
            return pts
    return 0


def points_i2(delta: float, s: float) -> int:
    if not math.isfinite(delta) or delta <= 0:
        return 0
    s = 0.0 if not math.isfinite(s) else s
    if delta > s and delta >= I2_MIN_DELTA:
        return 5
    return 4 if delta > s else 2


def points_i3(recalls_pct: dict[str, float]) -> int:
    if all(recalls_pct[c] >= ref for c, ref in HARD_CLASSES.items()):
        return 4
    low = min(recalls_pct.values())
    for floor, pts in HARD_FLOORS:
        if low >= floor:
            return pts
    return I3_MIN_POINTS


def cmd_grade(args) -> int:
    names = load_names(args.labels)
    hard_idx = hard_class_indices(names)
    final = load_group(args.final, args.test_csv, ref_what="test")
    base = load_group(args.baseline, args.test_csv, ref_what="test")
    if not args.test_csv:
        print("LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\n")
    warnings: list[str] = []
    for label, g in (("chung kết", final), ("mốc", base)):
        if len(g.preds) < MIN_SEEDS:
            warnings.append(f"nhóm {label} chỉ có {len(g.preds)} seed (< {MIN_SEEDS})")
    if set(final.preds[0].filenames) != set(base.preds[0].filenames):
        warnings.append("tập Filename của chung kết và mốc khác nhau")

    items: list[tuple[str, str, int | None, int, str]] = []  # (mã, tiêu chí, điểm, tối đa, ghi chú)

    # I1
    acc = final.summary["top1"][0] * 100
    items.append(("I1", "Top-1 accuracy test", points_i1(acc), 7, f"{acc:.2f}% (mean {len(final.preds)} seed)"))

    # I2
    mf_f, sd_f = final.summary["macro_f1"]
    mf_b, sd_b = base.summary["macro_f1"]
    delta = mf_f - mf_b
    s = max([v for v in (sd_f, sd_b) if math.isfinite(v)], default=math.nan)
    items.append(("I2", "Macro-F1 cải thiện so với mốc", points_i2(delta, s), 5,
                  f"final {mf_f:.4f}, mốc {mf_b:.4f}, Δ={delta:+.4f}, s={s:.4f}"))

    # I3
    rec_mean = final.summary["recall"][0] * 100
    recalls = {c: float(rec_mean[i]) for c, i in hard_idx.items()}
    note = ", ".join(f"{c} {recalls[c]:.1f}% (mốc {HARD_CLASSES[c]}%)" for c in HARD_CLASSES)
    items.append(("I3", "Recall hai lớp khó", points_i3(recalls), 4, note))

    # I4(a)
    if args.uncal:
        uncal = load_group(args.uncal, args.test_csv, ref_what="test")
        e_after, e_before = final.summary["ece"][0], uncal.summary["ece"][0]
        items.append(("I4a", "ECE sau TS < ECE trước", int(e_after < e_before), 1,
                      f"trước {e_before:.4f}, sau {e_after:.4f}"))
    else:
        items.append(("I4a", "ECE sau TS < ECE trước", None, 1, "chưa chấm được (thiếu --uncal)"))

    # I4(b)
    if args.final_val:
        fv = load_group(args.final_val, args.val_csv, ref_what="val")
        gap = abs(fv.summary["macro_f1"][0] - mf_f)
        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP,
                      int(gap <= I4_MAX_VAL_TEST_GAP), 1,
                      f"val {fv.summary['macro_f1'][0]:.4f}, test {mf_f:.4f}, chênh {gap:.4f}"))
    else:
        items.append(("I4b", "Chênh macro-F1 val/test <= %.2f" % I4_MAX_VAL_TEST_GAP, None, 1,
                      "chưa chấm được (thiếu --final-val)"))

    # I5
    if args.latency_p95_ms is None:
        items.append(("I5", "Cấu hình thời gian thực", None, 2, "chưa chấm được (thiếu --latency-p95-ms)"))
    else:
        ok = args.latency_p95_ms <= I5_P95_BUDGET_MS
        pts = 0 if not ok else (2 if args.latency_method == "proper" else 1)
        items.append(("I5", "Cấu hình thời gian thực", pts, 2,
                      f"p95 = {args.latency_p95_ms:.1f} ms (ngân sách {I5_P95_BUDGET_MS:.0f} ms), "
                      f"đo {'đúng cách' if args.latency_method == 'proper' else 'chưa đủ warmup/synchronize'}"))

    scored = [i for i in items if i[2] is not None]
    got = sum(i[2] for i in scored)
    max_scored = sum(i[3] for i in scored)
    lines = ["## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)", "",
             "| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |", "|---|---|---|---|---|"]
    for code, crit, pts, mx, note in items:
        lines.append(f"| {code} | {crit} | {'-' if pts is None else pts} | {mx} | {note} |")
    lines += ["", f"**Tổng các ý đã chấm: {got} / {max_scored}** (phần I tối đa 20)."]
    pending = [i[0] for i in items if i[2] is None]
    if pending:
        lines.append(f"Chưa chấm được: {', '.join(pending)}.")
    if warnings:
        lines += ["", "Cảnh báo:"] + [f"- {w}" for w in warnings]
    lines += ["", "Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I)."]
    print("\n".join(lines))

    if args.out:
        out = Path(args.out)
        out.mkdir(parents=True, exist_ok=True)
        (out / "grade_I.json").write_text(json.dumps({
            "items": [{"code": c, "criterion": cr, "points": p, "max": m, "note": n} for c, cr, p, m, n in items],
            "total": got, "max_scored": max_scored, "warnings": warnings}, indent=2, ensure_ascii=False))
    return 0


# --------------------------------------------------------------------------- #
# CLI
# --------------------------------------------------------------------------- #
def build_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(description="Đánh giá Lab Day 2 (DeepWeeds)")
    sub = ap.add_subparsers(dest="cmd", required=True)

    sc = sub.add_parser("score", help="tính chỉ số cho một nhóm file dự đoán (nhiều seed)")
    sc.add_argument("--pred", nargs="+", required=True, help="glob của file dự đoán")
    sc.add_argument("--test-csv", help="labels/test_subset0.csv để đối chiếu tên file và nhãn")
    sc.add_argument("--labels", help="labels/labels.csv để lấy tên lớp")
    sc.add_argument("--tag", default="score")
    sc.add_argument("--out", help="thư mục lưu JSON/CSV")
    sc.set_defaults(func=cmd_score)

    gr = sub.add_parser("grade", help="tự chấm phần I của RUBRIC")
    gr.add_argument("--final", nargs="+", required=True, help="glob dự đoán test của cấu hình chung kết")
    gr.add_argument("--baseline", nargs="+", required=True, help="glob dự đoán test của mốc (T00 + I00)")
    gr.add_argument("--uncal", nargs="+", help="glob dự đoán test của chung kết khi chưa temperature scaling")
    gr.add_argument("--final-val", nargs="+", help="glob dự đoán val của chung kết")
    gr.add_argument("--latency-p95-ms", type=float, help="p95 batch 1 của cấu hình thời gian thực (ms)")
    gr.add_argument("--latency-method", choices=["proper", "partial"], default="proper",
                    help="proper = có warmup và synchronize; partial = chưa đủ")
    gr.add_argument("--test-csv", help="labels/test_subset0.csv")
    gr.add_argument("--val-csv", help="labels/val_subset0.csv (đối chiếu --final-val)")
    gr.add_argument("--labels", help="labels/labels.csv")
    gr.add_argument("--out", help="thư mục lưu grade_I.json")
    gr.set_defaults(func=cmd_grade)
    return ap


def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    try:
        return args.func(args)
    except (ValueError, FileNotFoundError) as e:
        print(f"LỖI: {e}", file=sys.stderr)
        return 2


if __name__ == "__main__":
    sys.exit(main())


#### Tài liệu đọc hiểu kèm bài nộp

In [ ]:
%%writefile dochieu.md
# Đọc hiểu project Lab Day 2 — DeepWeeds

**Học viên:** Hoàng Văn Nam  
**Mã học viên:** 2A202602853

## 1. Project làm gì?

Project huấn luyện mô hình thị giác máy tính để nhận diện ảnh cỏ dại. Đầu vào là một ảnh RGB; đầu ra là xác suất của **9 lớp**: Chinee apple, Lantana, Parkinsonia, Parthenium, Prickly acacia, Rubber vine, Siam weed, Snake weed và Negative. `Negative` là nhóm ảnh không thuộc tám loài mục tiêu, không phải ảnh trống.

Bài lab không chỉ yêu cầu train một mô hình. Nó khảo sát ba nhóm lựa chọn: **kiến trúc mạng**, **công thức huấn luyện** và **cách suy luận**; sau đó đo chất lượng và tốc độ để chọn cấu hình phù hợp. Kết quả có thể hỗ trợ khảo sát cỏ dại hoặc lựa chọn mô hình cho hệ thống xử lý ảnh trên robot. Đây là mô hình phân loại toàn ảnh; nó không trả vị trí/bounding box của từng cây và chưa phải một hệ thống phun thuốc tự động.

Dataset DeepWeeds có **17.509 ảnh**, chia fold 0 cố định thành train 10.501, validation 3.501 và test 3.507 ảnh. Lớp Negative chiếm khoảng một nửa dữ liệu, nên accuracy cao chưa chắc nhận diện tốt các loài ít mẫu.

## 2. Cách chạy notebook trên Colab

1. Vào Google Colab, chọn **File → Upload notebook**, mở `code/lab_day2.ipynb`.
2. Chọn **Runtime → Change runtime type → GPU**. Notebook kiểm tra CUDA trước khi chạy thí nghiệm.
3. Chạy các ô từ trên xuống hoặc **Run all**. Cho phép kết nối Drive khi Colab hỏi để lưu checkpoint bền vững.
4. Giữ cấu hình mặc định: 12 epoch, batch 32, seed sàng lọc 0; chung kết dùng seed 0, 1, 2. Batch 32 giảm bộ nhớ so với batch 64 gợi ý của đề. Nếu cần batch 16, đổi **trước lần chạy đầu**, giữ cùng batch cho cả chiến dịch và ghi nhận trong báo cáo.
5. Kết quả mặc định lưu ở `MyDrive/DeepWeeds_Lab/2A202602853_Hoang_Van_Nam/`. Nếu mất phiên, mở lại notebook, chạy lại với cùng cấu hình. Các lượt đã hoàn tất được đọc lại; lượt đang train tiếp tục từ checkpoint epoch gần nhất.
6. Ô cuối tạo ZIP bài nộp, bỏ dataset và trọng số lớn. Có thể tải ZIP qua Colab hoặc lấy trực tiếp từ Drive.

Notebook **tự chứa code**: nó tạo các file Python trong `/content/deepweeds_lab/code/`, kèm `eval.py` nguyên bản. Bạn không cần clone GitHub hoặc upload thêm các module. Những file Python trong repo giúp đọc và sửa code thuận tiện; bản notebook nhúng cùng nội dung các file này.

Notebook được chuẩn bị để bạn thực thi trên Colab. **Chưa có kết quả thực nghiệm chung kết được tạo bằng việc chạy notebook trong lần bàn giao này.** Bảng Excel và báo cáo được sinh khi bạn chạy, không chứa số liệu chất lượng giả định.

## 3. Luồng xử lý tổng thể

```mermaid
flowchart LR
    A[Ảnh và CSV DeepWeeds] --> B[Kiểm tra split và EDA]
    B --> C[So sánh 5 backbone]
    C --> D[Ablation công thức train]
    D --> E[TTA / ensemble / calibration / latency trên val]
    E --> F[Khóa lựa chọn và train 3 seed]
    F --> G[Đánh giá test và eval.py]
    G --> H[Excel / report / biểu đồ / ZIP]
```

**Train** cập nhật trọng số. **Validation** chọn backbone, loss, augmentation, checkpoint, phương pháp suy luận và nhiệt độ calibration. **Test** chỉ dùng sau khi khóa toàn bộ lựa chọn; không dùng kết quả test để quay lại chỉnh công thức.

Đây là cách tránh rò rỉ dữ liệu: nếu chọn mô hình dựa trên test, test không còn là phép kiểm tra độc lập. Việc xem phân bố nhãn test để kiểm tra cấu trúc split không được dùng để tính class weights; class weights chỉ tính trên train.

## 4. Các file chính và tác dụng cụ thể

### `code/lab_day2.ipynb` — notebook bài làm

Đây là file bạn mở trên Colab. Nó chứa thông tin học viên, hướng dẫn chọn GPU, cấu hình đầu ra, cài thư viện, tạo mã nguồn, tải dataset, EDA, từng giai đoạn thí nghiệm và đóng gói bài nộp. Các ô thí nghiệm gọi cùng một pipeline, không copy train loop cho từng model. Notebook hiển thị bảng và ảnh để bạn kiểm tra tiến độ.

Khi chạy lại một notebook đã hoàn thành, các bước sàng lọc được bỏ qua nếu chiến dịch đã khóa lựa chọn test. Kết quả test đã lưu được dùng lại để xuất báo cáo. Nếu phiên bị ngắt giữa một lượt test trước khi cache được ghi thành công, lượt chưa hoàn tất có thể cần chạy lại; điều này không cho phép đổi cấu hình theo kết quả đã xem.

### `code/download_data.py` — lấy dữ liệu gốc

Tải `images.zip` từ Zenodo, kiểm MD5 đúng chuỗi quy định trong đề rồi giải nén 17.509 JPEG vào `data/images/`. Tải `labels.csv` và ba CSV fold 0 từ repo DeepWeeds của tác giả. File tải tạm dùng hậu tố `.part`; chỉ đổi thành tên chính sau khi tải xong.

Ảnh được ghi theo basename, không lấy đường dẫn trong ZIP làm đường dẫn đích; tên ảnh trùng bị từ chối. File đã tồn tại được tận dụng khi chạy lại. Lỗi tải mạng được thử lại tối đa ba lần, lỗi checksum không bị bỏ qua.

### `code/dataset.py` — split, Dataset và DataLoader

- `load_split()` đọc ba CSV cố định; không tự chia lại theo seed.
- `check_split()` kiểm nhãn 0–8, đủ lớp, tên file không trùng, giao các tập rỗng, hợp đủ 17.509 ảnh và ảnh tồn tại.
- `check_manifest()` đối chiếu nhãn split với `labels.csv`. Một sai lệch upstream đã biết được ghi nhận: `20170714-110407-3.jpg` có nhãn 0 trong train split và nhãn 1 trong manifest. Pipeline giữ nguyên nhãn split và từ chối sai lệch khác.
- `build_transforms()` tạo crop/flip, ColorJitter hoặc RandAugment cho train. Validation/test dùng resize và center crop xác định, không augmentation ngẫu nhiên.
- `DeepWeedsDataset` trả `(ảnh_tensor, nhãn, tên_file)`; tên file giúp giữ đúng thứ tự dự đoán.
- `make_loader()` tạo batch, seed worker, pin memory trên GPU; sampler cân bằng chỉ được bật cho train.

Chuẩn hóa dùng mean/std của trọng số tiền huấn luyện. Không ước lượng chuẩn hóa từ test.

### `code/model.py` — mạng và nhóm tham số

`build_model()` tạo backbone qua `timm` và thay classifier thành 9 lớp. Ba chế độ khởi tạo: `scratch` train từ đầu; `frozen` dùng pretrained và chỉ học head; `finetune` cập nhật cả backbone và head.

`freeze_backbone()` tắt gradient của backbone. `param_groups()` đặt LR riêng cho backbone/head; bias và tham số norm không chịu weight decay. `count_params()` đếm triệu tham số. `count_gmacs()` dùng fvcore đo chi phí tính toán; nếu có phép toán không hỗ trợ thì để kết quả chưa đo, không báo bằng 0. GMAC không thay thế latency thực tế.

### `code/losses.py` — hàm mất mát và trộn mẫu

Cross-entropy phạt xác suất thấp ở nhãn đúng. Label smoothing làm mục tiêu bớt tuyệt đối, có thể giảm quá tự tin. Focal loss giảm trọng số của mẫu dễ để tập trung mẫu khó; gamma = 0 trở về CE không trọng số. CE có trọng số tăng tác động của lớp ít mẫu, với trọng số tính từ số lượng ảnh **train**.

`mix_batch()` thực hiện Mixup hoặc CutMix và trả hai nhãn cùng tỷ lệ `lambda`. `mixed_loss()` tính tổng có trọng số của loss hai nhãn. CutMix dùng tỷ lệ diện tích thực sau khi cắt biên; nếu chỉ thay ảnh mà giữ nhãn cứng như cũ, quá trình học sẽ sai. Accuracy train với ảnh trộn không có ý nghĩa như accuracy thông thường; chất lượng được đo trên val sạch.

### `code/train.py` — vòng huấn luyện dùng chung

`Config` tập trung các tham số của một lần chạy. `run()` kiểm dữ liệu, tạo model/loaders/loss/optimizer, train từng epoch và lưu checkpoint tốt nhất theo **macro-F1 val**. Trường hợp hòa giữ checkpoint sớm hơn.

`train_one_epoch()` dùng AMP trên CUDA, warmup và cosine theo optimizer step, tùy chọn EMA. Nếu backbone frozen, BatchNorm/dropout backbone ở eval, chỉ head ở train. Khi AMP bỏ qua optimizer step do overflow, scheduler và EMA cũng không cập nhật theo bước đó.

`evaluate()` dùng eval và inference mode, giữ đúng thứ tự file, trả logits và loss. `plot_curves()` lưu loss train/val cùng macro-F1 val. `last.pt` lưu model, optimizer, scheduler, scaler, EMA, lịch sử và RNG để tiếp tục theo epoch. `best.pt` phục vụ suy luận. Resume từ epoch đã commit không bảo đảm tái lập từng bit giữa GPU/thư viện khác nhau hoặc khi đổi số worker.

`config.json` ghi cấu hình thực tế, preprocessing, pretrained tag, phiên bản thư viện, GPU, hash CSV và mã nguồn. `summary.json` chỉ xuất sau khi train đủ số epoch. Checkpoint resume có trạng thái Python/NumPy nên chỉ mở checkpoint tin cậy do chính pipeline này tạo.

### `code/inference.py` — dự đoán và calibration

`predict_logits()` chạy eval, tắt gradient và trả filename, nhãn, logits theo cùng thứ tự. Các phép `view_hflip`, `views_multicrop`, `views_multiscale` tạo cách nhìn khác nhau của cùng ảnh.

`aggregate_views(space='prob')` lấy trung bình softmax từng view. `space='logit'` lấy trung bình logits rồi mới softmax. Hai cách không tương đương nói chung. `ensemble_probs()` lấy trung bình xác suất nhiều model; các model phải dự đoán cùng ảnh theo cùng thứ tự.

`fit_temperature()` tối ưu một nhiệt độ dương để giảm NLL trên validation. `apply_temperature()` tính `softmax(logits / T)`. T dương giữ thứ tự logits nên giữ argmax; xác suất và ECE có thể thay đổi. Giảm NLL không bảo đảm ECE giảm. Với TTA/ensemble, log của xác suất tổng hợp đóng vai trò logits tương đương để calibration.

`fuse_conv_bn()` tạo bản sao model và gộp Conv–BatchNorm ở các cấu trúc được biết là liền nhau trong forward. Công thức sử dụng running mean/variance ở eval; activation đi kèm BatchNormAct của timm được giữ lại. Không tự gộp hai module chỉ vì chúng có tên gần nhau. Model LayerNorm như DeiT/ConvNeXt không áp dụng tối ưu BN.

### `code/benchmark.py` — đo tốc độ

`bench()` bỏ ít nhất 10 lượt warmup, đo ít nhất 50 lượt bằng `perf_counter`, đồng bộ GPU trước/sau mỗi lần đo rồi trả p50/p95/p99. p95 có nghĩa khoảng 95% lượt đo không vượt mức thời gian đó.

`latency_report()` ghi GPU, dtype, batch, kích thước ảnh, phiên bản torch và thông lượng. `tta_latency()` thực sự gọi model K lần; không lấy latency một view nhân K để giả làm số đo. Pipeline thí nghiệm đo riêng phương pháp suy luận đầy đủ trên tensor, gồm tạo view/gộp output; không tính đọc JPEG, PIL preprocessing hoặc I/O.

### `code/experiments.py` — điều phối bài lab

Bước 1 chạy 5 backbone: ResNet-50, ResNeXt-50, ConvNeXt-Tiny, DeiT-Small và MobileNetV3-Large. Cùng recipe, split, seed và số epoch. Tốt nhất về macro-F1 val được chọn sang bước sau.

Bước 2 lấy backbone đã chọn làm `T00` rồi thử các trục A–G: khởi tạo; augmentation/Mixup/CutMix; loss; sampler; LR; EMA; độ phân giải nếu backbone chấp nhận. Mỗi ablation thay một yếu tố. `T99_combined` kết hợp lựa chọn tốt trên từng trục rồi được train và đo lại, vì tác động có thể không cộng dồn. Mốc ResNet-50 ban đầu vẫn có ở `B01_resnet50`.

Bước 3 đo single view, lật ngang, five-crop, gộp logit, ensemble, BN fusion, resize và AMP khi phù hợp. ViT kích thước cố định bỏ qua resize tùy ý; model không BN bỏ qua fusion. Ít nhất single/hflip/five-crop/logit/ensemble vẫn có thể dùng với DeiT ở 224. Chọn bằng macro-F1 val, hòa thì lấy p95 thấp hơn.

Bước 4 train final và baseline với seed 0/1/2, khớp calibration từng seed trên val, ghi lựa chọn khóa vào `experiment_state.json`, rồi mới chạy test. Test cache giúp xuất lại kết quả mà không suy luận lại các lượt đã hoàn tất. Cấu hình hoặc hash code khác chiến dịch đã lưu sẽ bị chặn; muốn khảo sát độc lập phải dùng thư mục output mới.

### `code/deliverables.py` — sản phẩm bài nộp

Đọc state, prediction CSV và các kết quả đã đo để tạo **7 sheet**: `Summary`, `Backbones`, `Training`, `Inference`, `Final`, `PerClass`, `Latency`. Dòng chưa có dữ liệu không được điền metric giả. Các metric tính bằng `eval.py` gốc; mean/std nhiều seed dùng sample std, `ddof=1`.

File này cũng tạo biểu đồ accuracy–latency, confusion matrix, gallery ảnh test dự đoán sai có filename/nhãn/xác suất, `report.md` và README bài nộp. Báo cáo tự tổng hợp số liệu và nêu hạn chế; bạn nên đọc ảnh lỗi để bổ sung nhận xét thị giác nếu giảng viên yêu cầu phân tích sâu bằng mắt.

### `code/prepare.py`, `code/eda.py`, `code/preflight.py`

`eda.py` kiểm kích thước/kênh gốc và đọc xác minh ảnh, không fit preprocessing. `prepare.py` gọi EDA, vẽ phân bố và ít nhất 3 ảnh/lớp, hiển thị ảnh augmentation đã giải chuẩn hóa, ghi split checks. `preflight.py` overfit một batch nhỏ để phát hiện lỗi cơ bản của train loop.

Sanity check dùng 9 ảnh train thật, ResNet-18 scratch và đầu vào 64×64. Nếu không học được batch nhỏ hoặc loss ban đầu cách xa `ln(9)`, pipeline dừng trước thí nghiệm chính. **Sanity check không phải kết quả baseline, validation hoặc test.**

### `eval.py` — bộ đánh giá gốc của đề

Giữ nguyên file gốc. `save_predictions()` lưu `Filename`, `y_true`, `y_pred`, `p0..p8`; xác suất phải cộng thành 1 và `y_pred` phải là argmax. `score` đối chiếu nhãn/tên file với test CSV, tính metric từng seed và mean/std. `grade` tự chấm phần I theo ngưỡng tạm thời của RUBRIC, không thay kết luận của giảng viên.

Macro-F1 là trung bình F1 của 9 lớp, cho các lớp tầm quan trọng ngang nhau. Balanced accuracy là trung bình recall. ECE dùng 15 bin để đo lệch giữa confidence và tỷ lệ đúng. NLL đánh giá xác suất gán cho nhãn thật. Chỉ số nào cũng phải gắn với tập đo val hoặc test.

### File hướng dẫn và phụ trợ

| File/thư mục | Vai trò |
|---|---|
| `README.md` gốc | Đề bài, split, metric, yêu cầu bài nộp; giữ nguyên. |
| `GUIDE.md` | Quy trình thực nghiệm và câu hỏi phân tích; giữ nguyên. |
| `RUBRIC.md` | Tiêu chí chấm, ràng buộc và lỗi bị trừ điểm; giữ nguyên. |
| `starter/` | Bộ khung gốc còn stub; giữ nguyên để đối chiếu, không dùng để train bài hoàn chỉnh. |
| `code/README.md` | Điểm bắt đầu sử dụng notebook và các module đã hoàn thiện. |
| `code/requirements.txt` | Dependency tham khảo cho module Python. Notebook giữ cặp torch/torchvision sẵn có tương thích của Colab, cài thư viện còn lại. |
| `tests/test_eval.py` | Kiểm bộ đánh giá và hợp đồng dự đoán của đề. |
| `tests/test_starter.py` | Kiểm khung gốc vẫn là khung, notebook khung hợp lệ. |
| `tests/test_pipeline.py` | Kiểm dataset, loss, optimizer, overfit, resume và split. |
| `tests/test_inference.py` | Kiểm calibration, view, giữ thứ tự dự đoán, fusion và số lượt timing. |
| `tests/test_experiments.py` | Kiểm đủ backbone/trục/phương pháp và chỉ chọn bằng metric val. |
| `.gitignore` | Loại dataset, môi trường Python, checkpoint và file tạm khỏi Git. |
| `.venv/`, `__pycache__/` | Môi trường local và cache Python; không cần upload lên Colab. |
| `report.md` ở root | Báo cáo chuẩn bị có sẵn từ lần làm trước, chưa phải báo cáo chung kết mới. |
| `dochieu.md` | Tài liệu bạn đang đọc; cũng được nhúng vào notebook và copy vào bài nộp. |

## 5. File được sinh sau khi chạy

| Đường dẫn trong output | Nội dung và lúc tạo |
|---|---|
| `experiment_state.json` | Tiến độ chiến dịch, bảng kết quả, hash code, cấu hình và lựa chọn đã khóa. |
| `runs/<exp_id>/seed<k>/config.json` | Cấu hình, phiên bản, preprocessing, tag pretrained, GPU và provenance. |
| `runs/.../history.csv` | Mỗi epoch: train loss, val loss, macro-F1, accuracy, LR, thời gian. |
| `runs/.../last.pt` | Trạng thái đầy đủ để resume từ epoch đã ghi xong. |
| `runs/.../best.pt` | Model/EMA tốt nhất theo macro-F1 validation. |
| `runs/.../val_logits.npz` | Logits và filename/nhãn validation để đối chiếu. |
| `runs/.../summary.json` | Tóm tắt sau khi hoàn tất train đủ epoch. |
| `predictions/*_val.csv` | Xác suất validation, chưa dùng test để chọn cấu hình. |
| `predictions/F01_seed*_test.csv` | Dự đoán final đã calibration của ba seed. |
| `predictions/F01uncal_seed*_test.csv` | Bản chưa calibration để so ECE trước/sau. |
| `predictions/T00_seed*_test.csv` | Baseline cùng số seed với final. |
| `test_cache/seed*.npz` | Kết quả test đã suy luận để xuất lại mà không chạy model lại. |
| `curves/` | Training curves từng B/T/F, biểu đồ trade-off, confusion matrix và ảnh lỗi. |
| `artifacts/preparation/` | Counts, split checks, ảnh EDA/augmentation và sanity check. |
| `eval_out/` | JSON/CSV metric từng seed/tổng hợp, tự chấm và ảnh lỗi dưới dạng bảng. |
| `results.xlsx` | Workbook 7 sheet theo GUIDE. |
| `report.md`, `README.md` | Báo cáo đo thực tế và hướng dẫn tái lập của bài nộp. |
| `code/` và `eval.py` | Bản mã nguồn và notebook kèm theo bài nộp. |
| ZIP bài nộp | Chỉ lấy code, workbook, report, curves, predictions và metadata nhỏ; bỏ ảnh, checkpoint, logits/cache lớn. |

## 6. Đọc kết quả như thế nào?

Đầu tiên đọc `Summary`, sau đó đối chiếu `Backbones` và `Training`. Không chọn model chỉ vì accuracy cao; ưu tiên macro-F1 và F1 các lớp hiếm. Đọc `Inference` cùng `Latency`: TTA/ensemble có thể tăng chất lượng nhưng tốn nhiều lượt forward. Nếu triển khai giới hạn 30 hoặc 100 ms, chỉ xét cấu hình đo được đáp ứng p95 trên phần cứng tương ứng.

Ở `Final`, so mean ± std qua ba seed. Nếu chênh lệch nhỏ hơn std, chưa phân biệt được hai cấu hình. Độ trễ Colab chỉ mô tả phần cứng Colab, không chứng minh robot thực tế chạy cùng tốc độ. Score của bài báo dùng khoảng 100 epoch, nên không so ngang như một cam kết điểm số cho lab 12 epoch.

Cuối cùng đọc confusion matrix và gallery lỗi. Đây là bằng chứng về các cặp lớp nhầm lẫn; nhận xét về lá/nền/ánh sáng là giả thuyết cần kiểm tra ảnh, không tự động suy ra chỉ từ metric. Project giúp học cách xây dựng thí nghiệm đáng tin và chọn mô hình có đánh đổi phù hợp, không chỉ tạo một con số accuracy.


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
# Tải Zenodo, kiểm MD5, giải nén 17.509 ảnh và tải split fold 0 gốc.
from download_data import main as download_deepweeds
download_deepweeds()


In [ ]:
IMAGES_DIR, LABELS_DIR = WORK_DIR/"data/images", WORK_DIR/"data/labels"
ARTIFACTS=WORK_DIR/"artifacts/preparation"
ARTIFACTS.mkdir(parents=True,exist_ok=True)
STATE_PATH=OUTPUT_DIR/"experiment_state.json"
import dataset, train, inference, benchmark, experiments
from prepare import eda, pipeline_check
from eval import read_pred, compute_metrics

def load_state():
 return json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}
def run_stage(stage):
 if load_state().get("sealed") and stage in ("backbones","training","inference"):
  print(stage,": đã khóa lựa chọn, đọc lại kết quả cũ."); return
 subprocess.run([sys.executable,"-u","code/experiments.py","--stage",stage,
  "--output",str(OUTPUT_DIR),"--epochs",str(EPOCHS),"--batch-size",str(BATCH_SIZE),
  "--workers",str(NUM_WORKERS)],check=True)
def export_now(): run_stage("export")
def show_table(name):
 rows=load_state().get(name,[])
 if rows:
  display(pd.DataFrame(rows).drop(columns=[c for c in ("config","configuration") if c in rows[0]]))
 else: print(name,": chưa có số liệu.")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Giữ nguyên split; kiểm giao rỗng, hợp đủ, ảnh tồn tại, đếm lớp và metadata.
checks=eda(IMAGES_DIR,LABELS_DIR,ARTIFACTS)
splits=dataset.load_split(LABELS_DIR)
display(pd.read_csv(ARTIFACTS/"class_counts.csv"))
print("Split",checks["n"],"Union",checks["union"],"Overlap",checks["overlap"])
print("Kích thước",checks["sizes"],"Kênh",checks["channels"],"Tỷ lệ lớp lớn/nhỏ",checks["imbalance_ratio"])
print("Sai lệch upstream giữ nguyên:",checks["manifest_label_mismatches"])
display(Image(filename=str(ARTIFACTS/"class_distribution.png")))
display(Image(filename=str(ARTIFACTS/"class_samples.png")))
# Xem 3 ảnh/lớp: nền lá/cỏ, ánh sáng và che khuất có thể gây nhầm.
# Đây là giả thuyết thị giác, chưa phải kết luận confusion matrix.


In [ ]:
# Sanity check chỉ dùng 9 ảnh train thật, không coi là kết quả val/test.
torch.set_num_threads(2); train.set_seed(0)
sanity=pipeline_check(IMAGES_DIR,LABELS_DIR,ARTIFACTS,max_steps=100)
assert sanity["passed"], "Chưa overfit được batch nhỏ; không chạy thí nghiệm chính."
print({k:v for k,v in sanity.items() if k not in ("history","samples")})
display(Image(filename=str(ARTIFACTS/"augmented_samples.png")))
display(Image(filename=str(ARTIFACTS/"overfit_curve.png")))
import losses
z,y=torch.randn(9,9),torch.arange(9)
torch.testing.assert_close(losses.FocalLoss(0)(z,y),torch.nn.functional.cross_entropy(z,y))
torch.testing.assert_close(losses.LabelSmoothingCE(0)(z,y),torch.nn.functional.cross_entropy(z,y))
p=inference.apply_temperature(z.numpy(),2.)
assert np.allclose(p.sum(1),1.) and np.array_equal(p.argmax(1),z.numpy().argmax(1))
shutil.copytree(ARTIFACTS,OUTPUT_DIR/"artifacts/preparation",dirs_exist_ok=True)
print("Sanity check OK; không đưa các số này vào bảng kết quả thí nghiệm chính.")


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Cùng recipe/epoch/split/seed cho 5 backbone, chọn bằng val macro-F1.
run_stage("backbones"); show_table("Backbones")
selected=experiments.select_best(load_state()["Backbones"])
print("Backbone chọn:",selected["backbone"],"macro-F1 val:",selected["macro_f1_val"])
export_now()


## Bước 2 — Ablation công thức
T00 là baseline recipe trên backbone được chọn; B01 giữ mốc kiến trúc ResNet-50.
A: scratch/frozen/finetune; B: basic/color/RandAugment/Mixup/CutMix;
C: CE/smoothing/focal/weighted; D: sampler; E: LR; F: EMA; G: độ phân giải khi hỗ trợ.
Mỗi ablation thay một yếu tố. T99 kết hợp lựa chọn tốt từng trục rồi đo lại.


In [ ]:
run_stage("training"); show_table("Training")
selected=experiments.select_best(load_state()["Training"])
print("Recipe chọn:",selected["exp_id"],"macro-F1 val:",selected["macro_f1_val"])
print("Một seed sàng lọc: không kết luận ý nghĩa thống kê từ delta nhỏ.")
export_now()


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# VAL chỉ: 1-view/hflip/five-crop/logit/ensemble/fusion/resize/AMP khi phù hợp.
# Fit T trên val; latency 10 warmup + 50 lượt, synchronize, batch 1 và 32.
run_stage("inference"); show_table("Inference"); show_table("Latency")
export_now()
display(Image(filename=str(OUTPUT_DIR/"curves/inference_tradeoff.png")))
print("Latency forward trên tensor; không tính decode JPEG/PIL/I/O.")


## Bước 4 — Chung kết 3 seed, khóa lựa chọn rồi chạy test
Train final F01 và baseline T00 với seed 0/1/2; fit nhiệt độ từng seed trên val.
Khóa cấu hình trước lần test đầu; lưu final, uncalibrated, baseline và cache test.
Không đổi công thức sau khi xem test. Lượt đã hoàn tất được đọc lại khi resume.


In [ ]:
run_stage("final")
state=load_state()
assert state["sealed"]["selection_split"]=="val"
assert sorted(r["seed"] for r in state["Final"])==[0,1,2]
show_table("Final")
print("Lựa chọn đã khóa:",state["sealed"])


In [ ]:
# Đánh giá lại CSV bằng eval.py gốc, không chạy lại model trên test.
for tag in ("F01","T00"):
 subprocess.run([sys.executable,"eval.py","score","--pred",str(OUTPUT_DIR/f"predictions/{tag}_seed*_test.csv"),
  "--test-csv",str(LABELS_DIR/"test_subset0.csv"),"--labels",str(LABELS_DIR/"labels.csv"),
  "--tag",tag,"--out",str(OUTPUT_DIR/"eval_out")],check=True)


In [ ]:
lat=next(r for r in load_state()["Latency"] if r["exp_id"]=="F01" and r["batch"]==1)
subprocess.run([sys.executable,"eval.py","grade",
 "--final",str(OUTPUT_DIR/"predictions/F01_seed*_test.csv"),
 "--baseline",str(OUTPUT_DIR/"predictions/T00_seed*_test.csv"),
 "--uncal",str(OUTPUT_DIR/"predictions/F01uncal_seed*_test.csv"),
 "--final-val",str(OUTPUT_DIR/"predictions/F01_seed*_val.csv"),
 "--test-csv",str(LABELS_DIR/"test_subset0.csv"),"--val-csv",str(LABELS_DIR/"val_subset0.csv"),
 "--labels",str(LABELS_DIR/"labels.csv"),"--latency-p95-ms",str(lat["p95"]),
 "--latency-method","proper","--out",str(OUTPUT_DIR/"eval_out")],check=True)


## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
export_now()
expected={"Backbones","Training","Inference","Final","PerClass","Latency","Summary"}
assert set(pd.ExcelFile(OUTPUT_DIR/"results.xlsx").sheet_names)==expected
for sheet in ("Summary","Final","PerClass"):
 print(sheet); display(pd.read_excel(OUTPUT_DIR/"results.xlsx",sheet_name=sheet,header=3))
for tag in ("F01","T00"):
 assert len(list((OUTPUT_DIR/"predictions").glob(f"{tag}_seed*_test.csv")))==3
for row in load_state()["Backbones"]+load_state()["Training"]:
 assert (OUTPUT_DIR/"curves"/f"{row['exp_id']}_seed{row['seed']}.png").exists()
for seed in (0,1,2): assert (OUTPUT_DIR/"curves"/f"F01_seed{seed}.png").exists()
display(Image(filename=str(OUTPUT_DIR/"curves/F01_confusion.png")))
if (OUTPUT_DIR/"curves/F01_errors.png").exists(): display(Image(filename=str(OUTPUT_DIR/"curves/F01_errors.png")))
display(Markdown((OUTPUT_DIR/"report.md").read_text(encoding="utf-8")))
(OUTPUT_DIR/"environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True),encoding="utf-8")
# Lưu notebook đang mở vào bài nộp. Nếu frontend không cho phép, tải .ipynb qua menu File.
try:
 from google.colab import _message
 current=_message.blocking_request("get_ipynb",timeout_sec=30)["ipynb"]
 (OUTPUT_DIR/"code/lab_day2.ipynb").write_text(json.dumps(current,ensure_ascii=False,indent=1),encoding="utf-8")
except Exception as error:
 print("Dùng File → Download .ipynb rồi kèm notebook này vào bài nộp.",type(error).__name__)
import zipfile
zip_path=OUTPUT_DIR.parent/f"{STUDENT_ID}_Hoang_Van_Nam.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as archive:
 for file in sorted(OUTPUT_DIR.rglob("*")):
  if not file.is_file(): continue
  relative=file.relative_to(OUTPUT_DIR)
  if any(x in {"data","test_cache","__pycache__"} for x in relative.parts): continue
  if file.suffix in {".pt",".pth",".ckpt",".npz",".pyc",".tmp"}: continue
  archive.write(file,arcname=str(Path(OUTPUT_DIR.name)/relative))
print("ZIP:",zip_path,"— đọc report/README/dochieu và lấy link Colab bằng Share nếu cần.")
DOWNLOAD_ZIP=True
if DOWNLOAD_ZIP:
 from google.colab import files
 files.download(str(zip_path))
